# Multi-label classificatie van koraalcondities met deep learning

Auteur: Rainesh Rewat
Opleiding: Master Applied Artificial Intelligence, Hogeschool van Amsterdam
Contact: rainesh.rewat@hva.nl

## 1. Introductie

Koraalriffen bedekken maar een klein deel van de oceaanbodem. Toch leeft een groot deel van het zeeleven er in [1] [2]. Voor mensen aan de kust zijn koraalriffen ook belangrijk. Ze helpen bij visserij en toerisme en ze bieden ook bescherming tegen stormen [1].

Het gaat niet goed met veel koraalriffen. Stijgende watertemperaturen, verzuring, vervuiling en schadelijke visserij zetten riffen onder druk [1] en [2]. NOAA meldde dat meer dan 84% van de riffen wereldwijd geraakt is door de vierde wereldwijde bleachinggebeurtenis [3]. Bij bleaching verliest koraal de algen waarmee het samenleeft. Als de situatie niet verbetert sterft het koraal vaak af.

Onderzoekers gebruiken daarom steeds vaker onderwaterbeelden die door duikers of voertuigen onder water zijn gemaakt [4]. Dat levert veel beeldmateriaal op. Het handmatig nakijken kost alleen veel tijd. Voor 100 meter transect kan dat oplopen tot 80 minuten [5]. Classificatie van die beelden is inhoudelijk lastig. Lichtintensiteit wisselt sterk, beelden kunnen onscherp zijn door stromingen en klassen zoals dood koraal en puin lijken visueel op elkaar [5]. Deep learning kan helpen om dat werk sneller te doen. Omdat één koraalpatch meerdere condities tegelijk kan bevatten, past multi-label classificatie hier beter dan een gewone single-label aanpak [5].

## 2. Probleemstelling

### 2.1 Bestaand onderzoek

Er is al veel gedaan met beelddata voor koraalclassificatie. Oudere studies gebruikten handgemaakte kenmerken. Beijbom et al. combineerden kleur en textuur met een SVM en haalden 83% nauwkeurigheid [6]. Shihavuddin et al. gebruikten Gabor filters en grey level co-occurrence matrices met verschillende classifiers [7].

Later kwamen deep learning modellen erbij. Gomez-Rios et al. testten Inception v3 en ResNet en DenseNet op de EILAT en RSMAS datasets en lieten zien dat data augmentation de resultaten kan verbeteren [8]. Borbon et al. testten een eigen CNN op drie klassen en haalden 84,93% nauwkeurigheid [9].

Deze studies werken met single-label classificatie. Elk beeld krijgt dus maar één label, waardoor informatie over andere condities in hetzelfde beeld verloren gaat.

Het recente werk van Shao et al. pakt dit beter aan met multi-label classificatie. Zij maakten een dataset van 20.800 koraalbeelden uit Koh Tao in Thailand en testten zeven modelgroepen op vier condities en vier stressoren [5]. Hun ensemble haalde een macro F1 van 81.74% en een match ratio van 63.33%. In een vervolgstudie groeide de dataset naar 23.965 beelden en werden negen modellen getest. Daar haalde DINOv2 met LoRA een macro F1 van 83.79% [16].

### 2.2 Wat dit onderzoek toevoegt

Shao et al. keken vooral naar zo sterk mogelijke modellen zoals Swin Transformers en DINOv2. Minder duidelijk is hoe eenvoudigere en beter toegankelijke modellen het doen op precies dezelfde multi-label taak.

Dit project voegt daarom drie dingen toe. Eerst komt een CNN from scratch mee, zodat duidelijk wordt hoeveel ImageNet pretraining echt helpt. Ook komt per label een geoptimaliseerde drempel in plaats van overal 0.5. En GradCAM laat zien waar het model naar kijkt bij misclassificaties [19].

### 2.3 Onderzoeksvragen

De hoofdvraag is: hoe presteren een CNN from scratch en standaard transfer learning modellen op multi-label classificatie van koraalcondities?

De deelvragen zijn:

1. Welk transfer learning model presteert het best op deze taak: EfficientNet-B7, ResNet-101 of DenseNet-201?

2. Voor welke condities is classificatie het lastigst en waarom?

## 3. Setup en configuratie

In dit deel worden alle libraries geladen. De random seeds worden vastgezet en de GPU wordt ingesteld. Mixed precision en `channels_last` helpen om het geheugengebruik lager te houden en de training sneller te maken. Om reproduceerbaarheid te ondersteunen is `cudnn.benchmark` uitgeschakeld en `cudnn.deterministic` ingeschakeld, conform de PyTorch reproducibility-richtlijn [20].


In [ ]:
import os
import sys
import gc
import math
import time
import random
import hashlib
import zipfile
import subprocess
import urllib.request
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from collections import Counter
from pathlib import Path
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification"], check=True)
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and USE_AMP and not USE_BF16
PIN_MEMORY = torch.cuda.is_available()
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20

IMG_SIZE = 512
LR = 1e-4
WEIGHT_DECAY = 5e-4
NUM_EPOCHS = 30
PATIENCE = 5

NUM_WORKERS = min(8, max(os.cpu_count() or 2, 4)) if torch.cuda.is_available() else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None

gpu_total_gb = 0.0
if torch.cuda.is_available():
    gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_total_gb >= 80:
    # H100 / A100 80GB
    ARCH_START_BATCH = {
        "cnn": 256,
        "efficientnet_b7": 64,
        "resnet101": 96,
        "densenet201": 64,
    }
elif gpu_total_gb >= 40:
    # A100 40GB
    ARCH_START_BATCH = {
        "cnn": 192,
        "efficientnet_b7": 24,
        "resnet101": 64,
        "densenet201": 40,
    }
elif gpu_total_gb >= 20:
    # L4 / A10 / RTX 3090 (24GB)
    ARCH_START_BATCH = {
        "cnn": 128,
        "efficientnet_b7": 16,
        "resnet101": 32,
        "densenet201": 20,
    }
else:
    # T4 (16GB) — DenseNet en EfficientNet zijn geheugenintensief
    ARCH_START_BATCH = {
        "cnn": 64,
        "efficientnet_b7": 8,
        "resnet101": 16,
        "densenet201": 10,
    }

ImageFile.LOAD_TRUNCATED_IMAGES = True

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # benchmark=False en deterministic=True zorgen voor reproduceerbaarheid
    # ten koste van een kleine snelheidswinst. Conform PyTorch reproducibility docs.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_float32_matmul_precision("high")

if os.path.exists("/content"):
    BASE_DIR = "/content/coral_multilabel"
else:
    BASE_DIR = os.path.join(os.getcwd(), "coral_multilabel")
os.makedirs(BASE_DIR, exist_ok=True)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory_gb():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    alloc = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    return alloc, reserved

def print_gpu_memory(prefix="GPU geheugen"):
    if torch.cuda.is_available():
        alloc, reserved = gpu_memory_gb()
        print(f"{prefix} allocated={alloc:.2f} GB reserved={reserved:.2f} GB")

print(f"Device {DEVICE}")
print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")
print(f"Torchvision {torchvision.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU {props.name}")
    print(f"GPU geheugen totaal {props.total_memory / 1024**3:.1f} GB")
print(f"Werkmap {BASE_DIR}")
print(f"IMG_SIZE {IMG_SIZE}")
print(f"Split {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)}")
print(f"AMP {USE_AMP}")
print(f"NUM_WORKERS {NUM_WORKERS}")
print(f"PREFETCH_FACTOR {PREFETCH_FACTOR}")
print(f"Start batch sizes {ARCH_START_BATCH}")

## 4. Data downloaden en voorbereiden

De annotaties en de labelmapping komen direct van GitHub. De beeldpatches komen van Google Drive. Na het downloaden worden ZIP bestanden uitgepakt en ook RAR bestanden als die aanwezig zijn. Voor de split gebeuren alleen technische controles. Bestanden worden gekoppeld en exacte duplicaten worden verwijderd. Er worden hier dus nog geen voorbeeldbeelden bekeken en er gebeurt ook nog geen inhoudelijke analyse van labelverdelingen.


### Dataselectie

De keuze voor de Koh Tao Coral Condition Dataset [5] is gebaseerd op drie overwegingen. Het is veel groter dan eerdere datasets [8, 9]. Het ondersteunt multi-label classificatie [5]. En het is gepubliceerd in een peer-reviewed journal met expert-annotaties [5]. De data komt uit een enkele veldcontext (Koh Tao), wat de generaliseerbaarheid beperkt, maar de consistentie verhoogt.

In [ ]:
DATA_DIR = os.path.join(BASE_DIR, "data")
TAB_DIR = os.path.join(DATA_DIR, "tabular_data")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

csv_urls = {
    "annotations.csv": "https://raw.githubusercontent.com/XL-SHAO/CoralConditionDataset/main/tabular_data/annotations.csv",
    "labelsets.csv": "https://raw.githubusercontent.com/XL-SHAO/CoralConditionDataset/main/tabular_data/labelsets.csv",
    "surveys_metadata.csv": "https://raw.githubusercontent.com/XL-SHAO/CoralConditionDataset/main/tabular_data/surveys_metadata.csv",
}

for name, url in csv_urls.items():
    dst = os.path.join(TAB_DIR, name)
    if not os.path.exists(dst):
        print(f"Download van {name}")
        urllib.request.urlretrieve(url, dst)

try:
    import gdown
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

IMAGE_FOLDER_NAMES = ("clipped", "coral_image_external", "coral_images_dry", "coral_images_wet")

def count_images():
    return [p for p in Path(DATA_DIR).rglob("*.jpg")
            if p.parent.name.lower() in IMAGE_FOLDER_NAMES]

if len(count_images()) > 100:
    print(f"Beeldbestanden al aanwezig: {len(count_images())}")
else:
    print("Beelddata downloaden")

    DRIVE_FOLDER_ID = "1yjvVGSXuFRcO3b0SehyeAHzMtCHhI6S1"
    try:
        gdown.download_folder(id=DRIVE_FOLDER_ID, output=DATA_DIR, quiet=False, remaining_ok=True)
    except Exception as e:
        print(f"Folder download mislukt: {e}")

    IMAGE_FILES = [
        {"name": "coral_image_external.zip", "id": "1BJVJX-wJXyh2rBYK_jrKjsrxejS0159P"},
        {"name": "coral_images_dry.zip",     "id": "1QFqlCFL4cY8mn2yS6LPYJmjRcwRwSrAQ",
         "url": "https://drive.usercontent.google.com/download?id=1QFqlCFL4cY8mn2yS6LPYJmjRcwRwSrAQ&export=download&authuser=0"},
        {"name": "coral_images_wet.rar",     "id": "1cbmTZ9-y7B51PPFyStSZhJNgeqD7WTwA",
         "url": "https://drive.usercontent.google.com/download?id=1cbmTZ9-y7B51PPFyStSZhJNgeqD7WTwA&export=download&authuser=0"},
        {"name": "coral_images_extra_1.zip", "id": "1hAHywbY_8O8VGHu1kOYrABfzkNCzA8Kt"},
        {"name": "coral_images_extra_2.zip", "id": "1DbOd8PnUlNDsWSi1rxpWtacEyVNwfeV1"},
    ]

    for f in IMAGE_FILES:
        dst = os.path.join(DATA_DIR, f["name"])
        if os.path.exists(dst):
            print(f"Al aanwezig, overslaan: {f['name']}")
            continue
        print(f"Downloaden: {f['name']}")
        success = False

        if "url" in f:
            ret = subprocess.run(
                ["wget", "-q", "--show-progress", "-O", dst, f["url"]],
                check=False
            )
            if ret.returncode == 0 and os.path.exists(dst) and os.path.getsize(dst) > 1024:
                print(f"  OK via directe URL")
                success = True
            elif os.path.exists(dst):
                os.remove(dst)

        if not success:
            try:
                gdown.download(id=f["id"], output=dst, quiet=False)
                if os.path.exists(dst) and os.path.getsize(dst) > 1024:
                    print(f"  OK via gdown")
                    success = True
            except Exception as e:
                print(f"  Mislukt: {e}")

        if not success:
            print(f"  Overgeslagen - probeer later opnieuw")

zip_files = list(Path(DATA_DIR).rglob("*.zip"))
if zip_files:
    print(f"ZIP bestanden uitpakken: {len(zip_files)}")
    for zf in zip_files:
        if not zipfile.is_zipfile(zf):
            print(f"  Ongeldig ZIP-bestand (mislukte download?): {zf.name} — verwijderd")
            zf.unlink()
            continue
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(zf.parent)
        print(f"  Uitgepakt: {zf.name}")

rar_files = list(Path(DATA_DIR).rglob("*.rar"))
if rar_files:
    if shutil.which("unrar") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)
    print(f"RAR bestanden uitpakken: {len(rar_files)}")
    for rf in rar_files:
        subprocess.run(["unrar", "x", "-o+", str(rf), str(rf.parent)], check=False)

print(f"Totaal gevonden beelden: {len(count_images())}")


In [ ]:
IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png")
IMAGE_FOLDER_NAMES = (
    "clipped",
    "clipped_512",
    "coral_image_external",
    "coral_images_dry",
    "coral_images_wet",
)

In [ ]:
annotations_path = os.path.join(TAB_DIR, "annotations.csv")
labelsets_path = os.path.join(TAB_DIR, "labelsets.csv")
surveys_path = os.path.join(TAB_DIR, "surveys_metadata.csv")

ann = pd.read_csv(annotations_path, dtype={"patchid": str, "label": str})
labelsets = pd.read_csv(labelsets_path)
surveys_meta = pd.read_csv(surveys_path)

print("Tabellen geladen")
print(f"Rijen in annotations.csv: {len(ann)}")
print(f"Rijen in labelsets.csv: {len(labelsets)}")
print(f"Rijen in surveys_metadata.csv: {len(surveys_meta)}")

def parse_label_ids(x):
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return []
    return [part.strip() for part in x.split(",") if part.strip() != ""]

condition_id_map = {
    "1": "healthy",
    "2": "compromised",
    "3": "dead",
    "4": "rubble",
}
available_labels = ["healthy", "compromised", "dead", "rubble"]

manifest_path = os.path.join(BASE_DIR, "dataset_manifest_conditions_dedup_v3.csv")
hash_cache_path = os.path.join(BASE_DIR, "image_hash_cache.csv")
conflict_audit_path = os.path.join(BASE_DIR, "duplicate_label_conflicts_v3.csv")

if os.path.exists(manifest_path):
    df = pd.read_csv(manifest_path)
    print("Bestaand manifest geladen")
else:
    df = ann.copy()

    for label_id, label_name in condition_id_map.items():
        df[label_name] = df["label"].apply(lambda x: int(label_id in parse_label_ids(x)))

    image_paths = [
        p for ext in IMAGE_EXTENSIONS
        for p in Path(DATA_DIR).rglob(ext)
        if p.parent.name.lower() in IMAGE_FOLDER_NAMES
    ]
    image_map = {p.stem: str(p) for p in image_paths}

    df["filepath"] = df["patchid"].map(image_map)
    matched = int(df["filepath"].notna().sum())
    print(f"Gelabelde patches met gekoppeld beeld: {matched} van {len(df)}")

    df = df[df["filepath"].notna()].copy().reset_index(drop=True)

    def file_md5(path, chunk_size=1 << 20):
        h = hashlib.md5()
        with open(path, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()

    hash_map = {}
    if os.path.exists(hash_cache_path):
        hash_df = pd.read_csv(hash_cache_path)
        hash_map = dict(zip(hash_df["filepath"], hash_df["file_hash"]))

    unique_paths = df["filepath"].drop_duplicates().tolist()
    missing_paths = [p for p in unique_paths if p not in hash_map]

    if missing_paths:
        print(f"Bestandshashes berekenen voor {len(missing_paths)} beelden")
        for p in tqdm(missing_paths):
            hash_map[p] = file_md5(p)
        hash_df = pd.DataFrame({"filepath": list(hash_map.keys()), "file_hash": list(hash_map.values())})
        hash_df.to_csv(hash_cache_path, index=False)

    df["file_hash"] = df["filepath"].map(hash_map)

    dup_patchid = int(df["patchid"].duplicated().sum())
    dup_filepath = int(df["filepath"].duplicated().sum())
    dup_hash = int(df["file_hash"].duplicated().sum())

    print("Technische controle voor de split")
    print(f"Dubbele patchid: {dup_patchid}")
    print(f"Dubbele bestandskoppelingen: {dup_filepath}")
    print(f"Dubbele beeldhashes: {dup_hash}")

    conflict_mask = (
        df.groupby("file_hash")[available_labels]
        .nunique()
        .max(axis=1)
        .gt(1)
    )
    conflict_hashes = conflict_mask[conflict_mask].index.tolist()
    print(f"Hashgroepen met labelverschil: {len(conflict_hashes)}")

    if conflict_hashes:
        conflict_audit = (
            df[df["file_hash"].isin(conflict_hashes)][["file_hash", "patchid", "label"] + available_labels]
            .sort_values(["file_hash", "patchid"])
            .reset_index(drop=True)
        )
        conflict_audit.to_csv(conflict_audit_path, index=False)
        print("Audit van labelconflicten opgeslagen")

    before = len(df)

    # Conflicterende duplicaten worden uitgesloten: twee rijen met dezelfde hash
    # maar verschillende labels kunnen niet betrouwbaar worden samengevoegd.
    df = df[~df["file_hash"].isin(conflict_hashes)].copy()
    excluded_conflicts = before - len(df)
    print(f"Uitgesloten wegens labelconflict (zelfde hash, andere labels): {excluded_conflicts}")

    # Voor de overige duplicaten zijn de labels consistent; 'first' volstaat.
    agg = {
        "patchid": "first",
        "label": lambda s: "|".join(sorted(set(map(str, s)))),
        "filepath": "first",
    }
    for lbl in available_labels:
        agg[lbl] = "first"

    df = df.groupby("file_hash", as_index=False).agg(agg).reset_index(drop=True)

    removed = (before - excluded_conflicts) - len(df)
    print(f"Verwijderd op basis van exacte beeldduplicaten (consistente labels): {removed}")
    print(f"Over na deduplicatie: {len(df)}")

    df.to_csv(manifest_path, index=False)

for lbl in available_labels:
    df[lbl] = df[lbl].astype(int)

print(f"Annotaties totaal: {len(ann)}")
print(f"Unieke gelabelde clipped beelden: {len(df)}")


### Near-duplicaten verwijderen

Naast exacte duplicaten kunnen near-duplicaten in de dataset zitten. Dat zijn beelden die er bijna hetzelfde uitzien, maar niet byte-voor-byte gelijk zijn, bijvoorbeeld door lichte belichtingsverschillen of kleine rotaties bij herhaalde surveys. Ze zijn een risico voor leakage, omdat ze niet worden gevangen door een MD5-check. Als ze niet vóór de split worden verwijderd, kunnen vrijwel identieke beelden in zowel de trainingsset als de testset terechtkomen.

Near-duplicaten worden opgespoord met pHash (perceptual hash). pHash maakt een 64-bits vingerafdruk op basis van de frequentie-inhoud van het beeld via een discrete cosinus transformatie [26]. De hamming-afstand tussen twee hashes geeft aan hoeveel bits verschillen. Een drempel van 10 bits is gekozen op basis van de aanbeveling van Zauner [26]. Afstanden onder de 10 wijzen op visueel vrijwel identieke beelden, terwijl afstanden daarboven inhoudelijk verschillende beelden zijn.

De vergelijking is gevectoriseerd via numpy. De 64-bits hashes worden als binaire matrices opgeslagen en de hamming-afstanden worden per blok berekend via het dot-product: hamming(a,b) = sum(a) + sum(b) − 2·dot(a,b) voor binaire vectoren. Sorteren op integer-waarde van de hash werkt niet voor hamming-afstand, omdat hashes met één bitverschil ver uit elkaar kunnen liggen als integer.

In [ ]:
try:
    import imagehash
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)
    import imagehash

NEAR_DUP_THRESHOLD = 10
PHASH_CACHE = os.path.join(BASE_DIR, "phash_cache.csv")

filepaths = df["filepath"].tolist()

# Laad cache als die bestaat, anders opnieuw berekenen
if os.path.exists(PHASH_CACHE):
    phash_df = pd.read_csv(PHASH_CACHE)
    phash_map = dict(zip(phash_df["filepath"], phash_df["phash_str"]))
    missing = [p for p in filepaths if p not in phash_map]
else:
    phash_map = {}
    missing = filepaths

if missing:
    print(f"pHash berekenen voor {len(missing)} beelden...")
    for p in tqdm(missing):
        try:
            phash_map[p] = str(imagehash.phash(Image.open(p).convert("RGB")))
        except Exception:
            pass
    pd.DataFrame({"filepath": list(phash_map.keys()), "phash_str": list(phash_map.values())}).to_csv(PHASH_CACHE, index=False)
    print("pHash cache opgeslagen")
else:
    print(f"pHash cache geladen ({len(phash_map)} beelden)")

# Herstel imagehash objecten uit strings
phashes = {p: imagehash.hex_to_hash(h) for p, h in phash_map.items() if p in set(filepaths)}

# Gevectoriseerde O(n²) hamming-vergelijking via numpy dot-product
# hamming(a,b) = sum(a) + sum(b) - 2*dot(a,b) voor binaire vectoren
# Blokgewijze aanpak: vergelijk elk blok met zichzelf (upper triangle) en met alle
# volgende beelden. Zo worden alle paren (i<j) exact één keer vergeleken.
paths_list = [p for p in filepaths if p in phashes]
hash_bits = np.array([phashes[p].hash.flatten().astype(np.int16) for p in paths_list])  # (N, 64)
norms = hash_bits.sum(axis=1)  # (N,) — aantal bits per hash

BLOCK = 500  # (500, 24000) int16 = 24 MB per blok
paths_to_remove = set()
N = len(paths_list)

for start in range(0, N, BLOCK):
    end = min(start + BLOCK, N)
    block = hash_bits[start:end]       # (B, 64)
    block_norms = norms[start:end]     # (B,)
    B = len(block)

    # Within-block: vergelijk bovenste driehoek (j > i)
    dot_w = block @ block.T            # (B, B) — int via int16 matmul
    dist_w = (block_norms[:, None] + block_norms[None, :] - 2 * dot_w)  # (B, B)
    bi_arr, bj_arr = np.where((dist_w <= NEAR_DUP_THRESHOLD) & np.triu(np.ones((B, B), dtype=bool), k=1))
    for bi, bj in zip(bi_arr, bj_arr):
        gi, gj = start + bi, start + bj
        if paths_list[gi] not in paths_to_remove:
            paths_to_remove.add(paths_list[gj])

    # Cross-block: vergelijk dit blok met alle toekomstige beelden
    if end < N:
        future = hash_bits[end:]       # (F, 64)
        future_norms = norms[end:]     # (F,)
        dot_c = block @ future.T       # (B, F)
        dist_c = block_norms[:, None] + future_norms[None, :] - 2 * dot_c  # (B, F)
        bi_arr, fi_arr = np.where(dist_c <= NEAR_DUP_THRESHOLD)
        for bi, fi in zip(bi_arr, fi_arr):
            gi, gj = start + bi, end + fi
            if paths_list[gi] not in paths_to_remove:
                paths_to_remove.add(paths_list[gj])

n_before = len(df)
df = df[~df["filepath"].isin(paths_to_remove)].reset_index(drop=True)
n_after = len(df)

print(f"Beelden gecontroleerd: {N}")
print(f"Near-duplicaten verwijderd (hamming <= {NEAR_DUP_THRESHOLD}): {n_before - n_after}")
print(f"Over na near-deduplicatie: {n_after}")

## 5. Data split en datakwaliteit

De dataset wordt gesplitst in 70% train, 10% validatie en 20% test, conform Shao et al. [16]. Bij de group-aware aanpak worden groepen intact gehouden ten koste van exacte stratificatie. Als fallback wordt iteratieve multilabel-stratificatie gebruikt, die de labelverdeling in alle sets zo gelijk mogelijk houdt [17]. Waar mogelijk wordt een group-aware aanpak toegepast op basis van padstructuur en patchnaam om leakage te verminderen.

In [ ]:
set_seed()
from itertools import combinations

def _safe_parent_name(path_str, level_from_file=2):
    p = Path(path_str)
    return p.parents[level_from_file - 1].name if len(p.parents) >= level_from_file else ""

def add_group_features(df_in):
    df_local = df_in.copy()
    df_local["filepath"] = df_local["filepath"].astype(str)
    df_local["patchid"] = df_local["patchid"].astype(str)

    df_local["path_group_1"] = df_local["filepath"].apply(lambda p: _safe_parent_name(p, 2))
    df_local["path_group_2"] = df_local["filepath"].apply(lambda p: _safe_parent_name(p, 3))
    df_local["patch_prefix"] = (
        df_local["patchid"]
        .str.extract(r"^([^_-]+)", expand=False)
        .fillna(df_local["patchid"])
    )
    return df_local

def iter_group_candidates(df_in):
    df_local = add_group_features(df_in)
    col_lookup = {c.lower(): c for c in df_local.columns}

    candidate_sets = [
        ["surveyid", "transectid"],
        ["survey_id", "transect_id"],
        ["surveyid"],
        ["survey_id"],
        ["transectid"],
        ["transect_id"],
        ["siteid"],
        ["site_id"],
        ["site"],
        ["location"],
        ["reef"],
        ["path_group_2", "path_group_1"],
        ["path_group_1"],
        ["patch_prefix"],
    ]

    yielded = set()
    for cand in candidate_sets:
        actual = tuple(col_lookup[c] for c in cand if c in col_lookup)
        if not actual or actual in yielded:
            continue
        yielded.add(actual)

        group_series = df_local[list(actual)].fillna("NA").astype(str).agg("|".join, axis=1)
        nunique = group_series.nunique()

        if 3 <= nunique < len(df_local):
            yield group_series, "+".join(actual), nunique

def multilabel_stratified_split(df_in):
    y = df_in[available_labels].values
    idx = np.arange(len(df_in))

    rest_ratio = VAL_RATIO + TEST_RATIO
    msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=rest_ratio, random_state=SEED)
    train_idx, rest_idx = next(msss1.split(idx, y))

    y_rest = y[rest_idx]
    test_share_of_rest = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_share_of_rest, random_state=SEED)
    val_rel_idx, test_rel_idx = next(msss2.split(rest_idx, y_rest))

    val_idx = rest_idx[val_rel_idx]
    test_idx = rest_idx[test_rel_idx]
    return train_idx, val_idx, test_idx

def score_three_way_split(df_in, y, groups, train_idx, val_idx, test_idx):
    splits = {
        "train": np.asarray(train_idx, dtype=int),
        "val": np.asarray(val_idx, dtype=int),
        "test": np.asarray(test_idx, dtype=int),
    }

    if len(splits["train"]) == 0 or len(splits["val"]) == 0 or len(splits["test"]) == 0:
        return float("inf")

    global_rate = y.mean(axis=0)
    score = 0.0

    for split_name, target_ratio in [
        ("train", TRAIN_RATIO),
        ("val", VAL_RATIO),
        ("test", TEST_RATIO),
    ]:
        split_idx = splits[split_name]
        split_y = y[split_idx]

        size_error = abs(len(split_idx) / len(df_in) - target_ratio)
        label_error = np.abs(split_y.mean(axis=0) - global_rate).mean()
        missing_labels = int((split_y.sum(axis=0) == 0).sum())

        score += 25.0 * size_error + 10.0 * label_error + 2.5 * missing_labels

    train_groups = set(groups[splits["train"]])
    val_groups = set(groups[splits["val"]])
    test_groups = set(groups[splits["test"]])

    overlap_penalty = (
        len(train_groups & val_groups)
        + len(train_groups & test_groups)
        + len(val_groups & test_groups)
    )
    score += 1000.0 * overlap_penalty

    return score

def exhaustive_group_aware_split(df_in, group_series):
    y = df_in[available_labels].values.astype(int)
    groups = group_series.astype(str).values
    unique_groups = pd.Index(pd.unique(groups)).tolist()

    if len(unique_groups) > 10:
        return None

    group_to_idx = {g: np.where(groups == g)[0] for g in unique_groups}
    best = None
    best_score = float("inf")

    for train_size in range(1, len(unique_groups) - 1):
        for train_groups in combinations(unique_groups, train_size):
            remaining = [g for g in unique_groups if g not in train_groups]
            if len(remaining) < 2:
                continue

            for val_size in range(1, len(remaining)):
                for val_groups in combinations(remaining, val_size):
                    test_groups = [g for g in remaining if g not in val_groups]
                    if len(test_groups) == 0:
                        continue

                    train_idx = np.concatenate([group_to_idx[g] for g in train_groups])
                    val_idx = np.concatenate([group_to_idx[g] for g in val_groups])
                    test_idx = np.concatenate([group_to_idx[g] for g in test_groups])

                    score = score_three_way_split(df_in, y, groups, train_idx, val_idx, test_idx)

                    if score < best_score:
                        best_score = score
                        best = {
                            "train": train_idx,
                            "val": val_idx,
                            "test": test_idx,
                        }

    if best is None:
        return None

    return best["train"], best["val"], best["test"], best_score

def random_group_aware_split_search(df_in, group_series, n_trials=240):
    y = df_in[available_labels].values.astype(int)
    groups = group_series.astype(str).values
    best = None
    best_score = float("inf")

    for trial in range(n_trials):
        rs = SEED + trial

        try:
            gss1 = GroupShuffleSplit(
                n_splits=1,
                train_size=TRAIN_RATIO,
                test_size=VAL_RATIO + TEST_RATIO,
                random_state=rs,
            )
            train_idx, rest_idx = next(gss1.split(df_in, y, groups))

            gss2 = GroupShuffleSplit(
                n_splits=1,
                train_size=VAL_RATIO / (VAL_RATIO + TEST_RATIO),
                test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
                random_state=rs,
            )
            val_rel_idx, test_rel_idx = next(
                gss2.split(df_in.iloc[rest_idx], y[rest_idx], groups[rest_idx])
            )
        except ValueError:
            continue

        val_idx = rest_idx[val_rel_idx]
        test_idx = rest_idx[test_rel_idx]

        score = score_three_way_split(df_in, y, groups, train_idx, val_idx, test_idx)

        if score < best_score:
            best_score = score
            best = {
                "train": train_idx,
                "val": val_idx,
                "test": test_idx,
            }

    if best is None:
        return None

    return best["train"], best["val"], best["test"], best_score

def group_aware_split_search(df_in, group_series):
    unique_groups = group_series.astype(str).nunique()

    if unique_groups <= 10:
        result = exhaustive_group_aware_split(df_in, group_series)
        if result is not None:
            return result

    return random_group_aware_split_search(df_in, group_series)

selected_group_series = None
selected_group_strategy = None
selected_group_count = None
train_idx = val_idx = test_idx = None
split_score = None

for candidate_series, candidate_strategy, candidate_count in iter_group_candidates(df):
    print(f"Probeer group-aware split op basis van: {candidate_strategy}")
    print(f"Aantal unieke groepen: {candidate_count}")

    result = group_aware_split_search(df, candidate_series)
    if result is None:
        print("  Geen geldige split gevonden voor deze groepssleutel\n")
        continue

    c_train_idx, c_val_idx, c_test_idx, c_score = result
    print(f"  Score: {c_score:.4f}")

    if split_score is None or c_score < split_score:
        split_score = c_score
        train_idx, val_idx, test_idx = c_train_idx, c_val_idx, c_test_idx
        selected_group_series = candidate_series
        selected_group_strategy = candidate_strategy
        selected_group_count = candidate_count

if selected_group_series is not None:
    df["group_id"] = selected_group_series.astype(str)
    split_method = f"group-aware ({selected_group_strategy})"
    print(f"Gekozen groepssleutel: {selected_group_strategy}")
    print(f"Aantal unieke groepen: {selected_group_count}")
    print(f"Beste split score: {split_score:.4f}")
else:
    print("Geen bruikbare group-aware split gevonden")
    print("Fallback naar iteratieve multilabel-stratificatie")
    train_idx, val_idx, test_idx = multilabel_stratified_split(df)
    df["group_id"] = df["patchid"].astype(str)
    split_method = "multilabel-stratified fallback"

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"Splitmethode: {split_method}")
print(f"Train: {len(df_train)} ({len(df_train)/len(df)*100:.0f}%)")
print(f"Val:   {len(df_val)} ({len(df_val)/len(df)*100:.0f}%)")
print(f"Test:  {len(df_test)} ({len(df_test)/len(df)*100:.0f}%)")

if "group_id" in df.columns and "group-aware" in split_method:
    overlap_train_val = set(df_train["group_id"]) & set(df_val["group_id"])
    overlap_train_test = set(df_train["group_id"]) & set(df_test["group_id"])
    overlap_val_test = set(df_val["group_id"]) & set(df_test["group_id"])

print("\nLabelverdeling per split")
for name, subset in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    parts = [f"{l}={subset[l].mean():.3f}" for l in available_labels]
    print(f"  {name}: {', '.join(parts)}")

### Korte bespreking van de split

De split levert 16.816 trainingsbeelden, 2.237 validatiebeelden en 4.785 testbeelden op. De labelverdeling verschilt licht tussen de sets. In de testset komt "dead" minder voor (0.346) dan in de trainingsset (0.412) en "compromised" is lager (0.244 versus 0.300). Dat is een gevolg van de group-aware aanpak. Hele locatiegroepen gaan naar één set en sommige locaties hebben meer dood koraal dan andere.


### Datasetnormalisatie voor CNN from scratch

Voor pretrained modellen schrijft Torchvision voor dat preprocessing afhankelijk is van de gekozen gewichten [30]. De officiële EfficientNet-B7 gewichten zijn getraind met een ingangsresolutie van 600×600 pixels. In dit project wordt bewust 512×512 pixels gebruikt voor alle architecturen, zodat de vergelijking eerlijk verloopt en de geheugenbelasting beheersbaar blijft. Dit is een gecontroleerde experimentele keuze. Torchvision waarschuwt dat afwijkende preprocessing van invloed kan zijn op de nauwkeurigheid [30]. Dit effect maakt deel uit van de analyse. Voor de CNN from scratch bestaan die verwachtingen niet. Het netwerk leert alle filters opnieuw. Onderwaterbeelden zijn gemiddeld donkerder en blauwer dan ImageNet-foto's, dus dataset-specifieke statistieken geven een beter gecentreerde input en bevorderen stabiele convergentie. De statistieken worden uitsluitend berekend op de trainingsset, zodat geen informatie uit de validatie- of testset lekt.

In [ ]:
def compute_dataset_stats(filepaths, n_samples=1000, resize=64, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(filepaths), size=min(n_samples, len(filepaths)), replace=False)
    sampled = [filepaths[int(i)] for i in idx]
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    for p in tqdm(sampled, desc="Dataset stats berekenen"):
        try:
            arr = np.array(Image.open(p).convert("RGB").resize((resize, resize)), dtype=np.float32) / 255.0
            pixels = arr.reshape(-1, 3).astype(np.float64)
            channel_sum += pixels.sum(axis=0)
            channel_sq_sum += np.square(pixels).sum(axis=0)
            pixel_count += len(pixels)
        except Exception:
            pass
    if pixel_count == 0:
        raise ValueError("Geen leesbare trainingsbeelden voor normalisatie")
    mean = channel_sum / pixel_count
    variance = np.maximum(channel_sq_sum / pixel_count - np.square(mean), 0.0)
    std = np.sqrt(variance)
    if np.any(std <= 1e-8):
        raise ValueError("Een kleurkanaal heeft geen variatie; normalisatie is niet mogelijk")
    return mean.tolist(), std.tolist()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DATASET_MEAN, DATASET_STD = compute_dataset_stats(df_train["filepath"].tolist())

print(f"Dataset  mean: {[round(v, 4) for v in DATASET_MEAN]}")
print(f"Dataset  std:  {[round(v, 4) for v in DATASET_STD]}")
print(f"ImageNet mean: {IMAGENET_MEAN}")
print(f"ImageNet std:  {IMAGENET_STD}")

## 6. Verkennende data-analyse

De EDA gebeurt alleen op de trainingsset. Zo wordt geen informatie uit de validatie of de test gebruikt tijdens het bouwen van het model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lc = df_train[available_labels].sum()
axes[0].bar(lc.index, lc.values, color=['#2ecc71','#f39c12','#e74c3c','#95a5a6'])
axes[0].set_title('Aantal beelden per conditielabel (train)')
axes[0].set_ylabel('Aantal')
for i, v in enumerate(lc.values):
    axes[0].text(i, v+50, str(int(v)), ha='center')

lpi = df_train[available_labels].sum(axis=1)
axes[1].hist(lpi, bins=range(0,6), edgecolor='black', color='#3498db', align='left')
axes[1].set_title('Aantal labels per beeld (train)')
axes[1].set_xlabel('Aantal labels'); axes[1].set_ylabel('Aantal beelden')
plt.tight_layout(); plt.show()
print(f'Gemiddeld labels per beeld: {lpi.mean():.2f}')

### Korte bespreking van de labelverdeling

"Healthy" is het meest voorkomende label met 12.107 trainingsbeelden. "Rubble" is het zeldzaamst met 1.851 beelden. Gemiddeld heeft een beeld 1.54 labels, wat bevestigt dat multi-label classificatie hier de juiste aanpak is. De verhouding healthy:rubble is ongeveer 6.5:1, wat een duidelijke positief-negatief onbalans laat zien.


In [ ]:
co = df_train[available_labels].T.dot(df_train[available_labels])
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(co, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Co-occurrence (train)')
plt.tight_layout(); plt.show()

### Korte bespreking van labelcombinaties

De co-occurrence matrix telt hoe vaak twee labels samen voorkomen in hetzelfde beeld. Dit wordt alleen op de trainingsset berekend, omdat je geen informatie uit de validatie- of testset mag gebruiken. De matrix helpt om te begrijpen welke labelcombinaties het model moet leren herkennen. Als "healthy" en "dead" vaak samen voorkomen, betekent dat dat het model moet leren om beide condities tegelijk te detecteren in dezelfde patch. Dit is typisch voor multi-label problemen, waar labels niet onafhankelijk van elkaar zijn [13]. Healthy+healthy komt het meest voor met 12107 labels. Het minst met 180 labels komt rubble+compromised voor.

In [ ]:
fig, axes = plt.subplots(len(available_labels), 4, figsize=(16, 4*len(available_labels)))
for row, label in enumerate(available_labels):
    sub = df_train[df_train[label]==1].sample(min(4, len(df_train[df_train[label]==1])), random_state=SEED)
    for col, (_, s) in enumerate(sub.iterrows()):
        img = Image.open(s['filepath']).convert('RGB')
        axes[row,col].imshow(img); axes[row,col].axis('off')
        others = [l for l in available_labels if s[l]==1 and l!=label]
        axes[row,col].set_title(label + (f' + {", ".join(others)}' if others else ''), fontsize=9)
plt.suptitle('Voorbeeldbeelden per conditielabel (train)', fontsize=14)
plt.tight_layout(); plt.show()

### Kleur- en helderheidsanalyse per label

Modellen kunnen achtergrondkenmerken leren in plaats van het koraal zelf [19]. Als healthy structureel helderder is dan dead, dan kan een model dat onderscheid maken op globale pixelwaarden zonder naar de koraaltextuur te kijken. De verdelingen hieronder geven een eerste kwantitatief beeld van dit risico. Een volledig uitsluitsel vereist achtergrond-segmentatie, wat buiten de scope van dit project valt.

In [ ]:
sample_per_label = 300
fig, axes = plt.subplots(2, len(available_labels), figsize=(5 * len(available_labels), 8))

for col, label in enumerate(available_labels):
    sub = df_train[df_train[label] == 1].sample(
        min(sample_per_label, (df_train[label] == 1).sum()), random_state=SEED
    )

    brightnesses = []
    ch_means = {c: [] for c in ['R', 'G', 'B']}

    for _, row in sub.iterrows():
        arr = np.array(Image.open(row['filepath']).convert('RGB').resize((64, 64)), dtype=float)
        brightnesses.append(arr.mean())
        for ci, c in enumerate(['R', 'G', 'B']):
            ch_means[c].append(arr[:, :, ci].mean())

    axes[0, col].hist(brightnesses, bins=30, color='#3498db', edgecolor='white')
    axes[0, col].set_title(f'{label}\nHelderheid')
    axes[0, col].set_xlabel('Gemiddelde pixelwaarde')

    for c, clr in zip(['R', 'G', 'B'], ['red', 'green', 'blue']):
        axes[1, col].hist(ch_means[c], bins=30, alpha=0.5, color=clr, label=c)
    axes[1, col].set_title(f'{label}\nKleurkanalen')
    axes[1, col].set_xlabel('Gemiddelde waarde')
    axes[1, col].legend(fontsize=7)

plt.suptitle('Helderheid en kleur per conditielabel (trainset, steekproef)', fontsize=12)
plt.tight_layout()
plt.show()

### Bespreking van helderheid en kleur

De helderheidshistogrammen overlappen sterk tussen de vier labels. De pieken liggen rond dezelfde waarde en de spreiding is vergelijkbaar. "Healthy" ligt gemiddeld iets hoger dan "dead" en "rubble", maar het verschil is klein ten opzichte van de spreiding binnen elk label. De RGB-verdelingen laten hetzelfde beeld zien, met alleen in het blauwe kanaal een lichte verschuiving voor "healthy". Dit is gunstig. Een model kan deze labels niet scheiden op puur globale kleurstatistiek en moet dus textuur en structuur gebruiken [8]. Het risico dat achtergrondpatronen worden meegeleerd blijft wel bestaan, omdat beelden van dezelfde duiklocatie vergelijkbare belichting delen [19].

## 7. Preprocessing en DataLoaders

De beelden worden op 512 bij 512 pixels gehouden, de originele patchgrootte van de dataset [5, 16]. In de ablationstudie wordt ook 224 bij 224 getest.

### Verliesfunctie

Asymmetric Loss (ASL) wordt gebruikt als verliesfunctie [25]. ASL behandelt positieve en negatieve samples asymmetrisch, wat direct aansluit op de labelonbalans in deze dataset. Focal loss [24] is symmetrisch en daardoor niet optimaal voor multi-label classificatie. Oversampling vergroot de trainingsset kunstmatig en verhoogt het risico op overfitting. ASL haalt aantoonbaar betere resultaten op multi-label benchmarks [25]. De standaard hyperparameters worden gebruikt: γ+=0, γ−=2 en margin=0.05, conform de aanbevelingen van Ridnik et al. [25].

In [ ]:
def make_transforms(img_size, train=False, aug=False, mean=None, std=None):
    if mean is None:
        mean = IMAGENET_MEAN
    if std is None:
        std = IMAGENET_STD
    ops = [transforms.Resize((img_size, img_size))]
    if train and aug:
        ops.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ])
    ops.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return transforms.Compose(ops)


class CoralMultiLabelDataset(Dataset):
    def __init__(self, dataframe, label_columns, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.label_columns = label_columns
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row[self.label_columns].values.astype(float), dtype=torch.float32)
        return img, labels


def make_datasets(img_size=IMG_SIZE, aug=False, mean=None, std=None):
    train_transform = make_transforms(img_size, train=True, aug=aug, mean=mean, std=std)
    eval_transform = make_transforms(img_size, train=False, aug=False, mean=mean, std=std)

    train_dataset = CoralMultiLabelDataset(df_train, available_labels, train_transform)
    val_dataset = CoralMultiLabelDataset(df_val, available_labels, eval_transform)
    test_dataset = CoralMultiLabelDataset(df_test, available_labels, eval_transform)
    return train_dataset, val_dataset, test_dataset, eval_transform


def _loader_kwargs(batch_size, shuffle):
    kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return kwargs


def make_loaders(train_batch_size, eval_batch_size=None, img_size=IMG_SIZE, aug=False, mean=None, std=None):
    if eval_batch_size is None:
        eval_batch_size = min(max(train_batch_size * 2, train_batch_size), 128)

    train_dataset, val_dataset, test_dataset, eval_transform = make_datasets(img_size, aug=aug, mean=mean, std=std)
    train_loader = DataLoader(train_dataset, **_loader_kwargs(train_batch_size, shuffle=True))
    val_loader = DataLoader(val_dataset, **_loader_kwargs(eval_batch_size, shuffle=False))
    test_loader = DataLoader(test_dataset, **_loader_kwargs(eval_batch_size, shuffle=False))
    return train_loader, val_loader, test_loader, eval_transform

## 8. Evaluatie en trainingsfuncties

De trainingsfuncties gebruiken mixed precision en `channels_last` zodat training op de GPU sneller kan gaan. Early stopping stopt de training als de validatie-macro F1 vijf epochs lang niet verbetert. De learning rate scheduler gebruikt CosineAnnealingLR, de cosine-annealingvariant uit het SGDR-paper van Loshchilov en Hutter [29], maar zonder de warm restarts die in dat paper centraal staan. De learning rate daalt vloeiend van de startwaarde naar een minimum van 1e-6 over de volledige trainingsperiode. In de praktijk wordt warmup gebruikt om te voorkomen dat grote gradiënten van de willekeurig geïnitialiseerde classificatielaag de pretrained backbone in de eerste epochs te sterk verstoren. Gradiënten worden geclipped op max_norm=1.0 via `clip_grad_norm_` na het unscalen van de GradScaler, conform Pascanu et al. [28] en de PyTorch AMP documentatie [22].

Macro F1 is de hoofdmaat voor modelselectie. Het berekent de F1 per label apart en neemt daarvan het gemiddelde, zodat elk label evenveel gewicht krijgt [5]. Zo tellen zeldzame labels zoals rubble even zwaar mee als veelvoorkomende labels. Match ratio wordt ook gerapporteerd als maat op beeldniveau. Dat is het percentage beelden waarbij de volledige voorspelling exact klopt [5].

### Modelvereisten

Het eindmodel moet aan de volgende meetbare vereisten voldoen. Deze worden na de finale evaluatie automatisch gecontroleerd.

1. Macro F1 op de testset hoger dan het null model.
2. Macro F1 op de testset hoger dan de LR baseline.
3. Alle vier de labels halen een F1 boven 0.00 op de testset.
4. Het verschil tussen de macro F1 op de trainingsset en de validatieset is kleiner dan 0.20.
5. De standaarddeviatie van macro F1 over drie seeds is kleiner dan 0.05 voor beide modellen.
6. De geoptimaliseerde drempels verbeteren de macro F1 op de validatieset ten opzichte van een vaste drempel van 0.5.

In [ ]:
class AsymmetricLoss(torch.nn.Module):
    """Asymmetric Loss for multi-label classification.
    Ridnik et al., ICCV 2021. arXiv:2009.14119.
    gamma_pos=0, gamma_neg=2, clip=0.05 zijn de standaard waarden uit het paper.
    Probability shift: p_m = max(p - m, 0) zodat negatieve kansen naar beneden worden verschoven.
    """
    def __init__(self, gamma_pos=0, gamma_neg=2, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        # Cast naar float32 zodat log-berekeningen niet underflowen in float16
        logits = logits.float()
        targets = targets.float()
        probs = torch.sigmoid(logits)
        probs_neg = probs if self.clip is None else torch.clamp(probs - self.clip, min=0)

        loss_pos = targets * torch.log(probs.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log((1 - probs_neg).clamp(min=self.eps))

        if self.gamma_pos > 0 or self.gamma_neg > 0:
            pt_pos = probs
            pt_neg = 1 - probs_neg
            loss_pos = loss_pos * ((1 - pt_pos) ** self.gamma_pos)
            loss_neg = loss_neg * ((1 - pt_neg) ** self.gamma_neg)

        loss = -(loss_pos + loss_neg)
        return loss.mean()


def make_criterion():
    return AsymmetricLoss(gamma_pos=0, gamma_neg=2, clip=0.05).to(DEVICE)

def autocast_context():
    if DEVICE.type == "cuda" and USE_AMP:
        amp_dtype = torch.bfloat16 if USE_BF16 else torch.float16
        return torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=True)
    return nullcontext()

def move_images(imgs):
    imgs = imgs.to(DEVICE, non_blocking=True)
    if imgs.ndim == 4 and DEVICE.type == "cuda":
        imgs = imgs.contiguous(memory_format=torch.channels_last)
    return imgs

def move_labels(labels):
    return labels.to(DEVICE, non_blocking=True)

def summarize_predictions(all_targets, all_preds, label_names, avg_loss=np.nan, all_probs=None):
    all_targets = np.asarray(all_targets).astype(int)
    all_preds = np.asarray(all_preds).astype(int)

    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    match_ratio = np.mean(np.all(all_preds == all_targets, axis=1))

    per_label = {}
    for i, lbl in enumerate(label_names):
        per_label[lbl] = {
            "precision": precision_score(all_targets[:, i], all_preds[:, i], zero_division=0),
            "recall": recall_score(all_targets[:, i], all_preds[:, i], zero_division=0),
            "f1": f1_score(all_targets[:, i], all_preds[:, i], zero_division=0),
        }

    result = {
        "loss": float(avg_loss) if not pd.isna(avg_loss) else np.nan,
        "macro_f1": macro_f1,
        "match_ratio": match_ratio,
        "per_label": per_label,
        "all_preds": all_preds,
        "all_targets": all_targets,
    }
    if all_probs is not None:
        result["all_probs"] = np.asarray(all_probs)
    return result

def evaluate_hard_predictions(y_true, y_pred, label_names):
    return summarize_predictions(y_true, y_pred, label_names, avg_loss=np.nan, all_probs=None)

def evaluate(model, loader, criterion, label_names, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_targets = [], [], []

    if np.isscalar(threshold):
        thr = np.full(len(label_names), float(threshold), dtype=np.float32)
    else:
        thr = np.asarray(threshold, dtype=np.float32)

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = move_images(imgs)
            labels = move_labels(labels)

            with autocast_context():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs).detach().float().cpu().numpy()
            preds = (probs >= thr.reshape(1, -1)).astype(int)
            targets = labels.detach().cpu().numpy().astype(int)

            total_loss += loss.item() * imgs.size(0)
            all_probs.append(probs)
            all_preds.append(preds)
            all_targets.append(targets)

    all_probs = np.vstack(all_probs)
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    avg_loss = total_loss / len(loader.dataset)
    return summarize_predictions(all_targets, all_preds, label_names, avg_loss=avg_loss, all_probs=all_probs)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, labels in loader:
        imgs = move_images(imgs)
        labels = move_labels(labels)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        probs = torch.sigmoid(outputs).detach().float().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        targets = labels.detach().cpu().numpy().astype(int)

        total_loss += loss.item() * imgs.size(0)
        all_preds.append(preds)
        all_targets.append(targets)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    return avg_loss, macro_f1

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                label_names, n_epochs=NUM_EPOCHS, patience=PATIENCE, save_path=None):
    best_f1 = -1.0
    best_state = None
    no_improve = 0
    amp_device = "cuda" if DEVICE.type == "cuda" else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=USE_FP16)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1": [],
        "val_f1": [],
    }

    if DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    for epoch in range(n_epochs):
        train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_result = evaluate(model, val_loader, criterion, label_names)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_result["loss"])
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_result["macro_f1"])

        if scheduler is not None:
            scheduler.step()

        improved = val_result["macro_f1"] > best_f1
        if improved:
            best_f1 = val_result["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        print(
            f"Epoch {epoch+1:02d}/{n_epochs}  "
            f"train_loss={train_loss:.4f}  val_loss={val_result['loss']:.4f}  "
            f"train_F1={train_f1:.4f}  val_F1={val_result['macro_f1']:.4f}"
        )

        if no_improve >= patience:
            print(f"Early stopping na epoch {epoch+1}")
            break

    if best_state is None:
        raise RuntimeError("Geen modelstatus opgeslagen tijdens training")

    model.load_state_dict(best_state)

    if save_path is not None:
        torch.save(best_state, save_path)
        print(f"Model opgeslagen: {save_path}")

    print_gpu_memory("Na training")
    return model, history

def plot_curves(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], label="Train loss")
    axes[0].plot(history["val_loss"], label="Val loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{title} - loss")
    axes[0].legend()

    axes[1].plot(history["train_f1"], label="Train macro F1")
    axes[1].plot(history["val_f1"], label="Val macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].set_title(f"{title} - macro F1")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def print_results(result, label_names, title):
    print(f"\n{title}")
    if not pd.isna(result.get("loss", np.nan)):
        print(f"  Loss:         {result['loss']:.4f}")
    print(f"  Match ratio: {result['match_ratio']:.4f}")
    print(f"  Macro F1:    {result['macro_f1']:.4f}")
    print("  Per label:")
    for lbl in label_names:
        m = result["per_label"][lbl]
        print(f"    {lbl:<15} P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}")

def tune_thresholds_per_label(y_true, y_prob, label_names, grid=None):
    if grid is None:
        grid = np.linspace(0.10, 0.90, 17)
    # Gebruik dezelfde float32-precisie als evaluate en behoud 0.5 als kandidaat.
    grid = np.unique(np.append(np.asarray(grid, dtype=np.float32), np.float32(0.5)))
    if not np.all(np.isfinite(grid)) or np.any((grid < 0) | (grid > 1)):
        raise ValueError("Drempels moeten eindige waarden tussen 0 en 1 zijn")
    y_prob = np.asarray(y_prob, dtype=np.float32)
    y_true = np.asarray(y_true)

    best_thresholds = []
    rows = []

    for i, lbl in enumerate(label_names):
        best_thr = 0.5
        best_f1 = f1_score(y_true[:, i], (y_prob[:, i] >= best_thr).astype(int), zero_division=0)
        for thr in grid:
            pred = (y_prob[:, i] >= thr).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = thr

        best_thresholds.append(best_thr)
        rows.append({"label": lbl, "best_threshold": best_thr, "val_f1": best_f1})

    return np.array(best_thresholds, dtype=np.float32), pd.DataFrame(rows)

def estimate_train_batch_size(build_fn, arch_name, num_labels, start_batch=None, safety_factor=0.90, img_size=None):
    if DEVICE.type != "cuda":
        return 8

    img_size = int(img_size or IMG_SIZE)
    start_batch = int(start_batch or ARCH_START_BATCH.get(arch_name, 16))
    batch_size = start_batch

    while batch_size >= 2:
        clear_gpu()
        model = None
        optimizer = None
        scaler = None
        dummy_x = None
        dummy_y = None
        loss = None
        try:
            model = build_fn(arch_name, num_labels)
            model = model.to(memory_format=torch.channels_last)
            model.train()

            criterion = make_criterion()
            optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            amp_device = "cuda" if DEVICE.type == "cuda" else "cpu"
            scaler = torch.amp.GradScaler(amp_device, enabled=USE_FP16)

            dummy_x = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)
            dummy_x = dummy_x.contiguous(memory_format=torch.channels_last)
            dummy_y = torch.zeros(batch_size, num_labels, device=DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                out = model(dummy_x)
                loss = criterion(out, dummy_y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()

            safe_batch = max(2, int(batch_size * safety_factor))
            print(f"Batchgrootte voor {arch_name} @ {img_size}px: probe={batch_size}, veilig={safe_batch}")
            return safe_batch

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"OOM bij batch {batch_size} voor {arch_name} @ {img_size}px, probeer kleiner")
                batch_size = batch_size // 2
            else:
                raise
        finally:
            del model, optimizer, scaler, dummy_x, dummy_y, loss
            clear_gpu()

    return 2

## 9. Null model

Het null model voorspelt altijd de meest voorkomende labelcombinatie uit de trainingsset. Dit is de absolute ondergrens. Een model dat hier niet overheen komt heeft in de praktijk weinig waarde.


In [ ]:
results_val = {}
results_test = {}

train_labels = df_train[available_labels].values
combos = ['_'.join(map(str, row)) for row in train_labels]
mc = Counter(combos).most_common(1)[0]
null_pred = np.array([int(x) for x in mc[0].split('_')], dtype=int)

print(f'Meest voorkomende labelcombinatie: {dict(zip(available_labels, null_pred.tolist()))} ({mc[1]}/{len(df_train)})')

val_labels = df_val[available_labels].values
null_val_preds = np.tile(null_pred, (len(val_labels), 1))
null_val_result = evaluate_hard_predictions(val_labels, null_val_preds, available_labels)

results_val['Null model'] = {
    'macro_f1': null_val_result['macro_f1'],
        'match_ratio': null_val_result['match_ratio'],
    'set': 'val',
}
print_results(null_val_result, available_labels, "Null model (validatie)")

### Bespreking van het null model

Het null model voorspelt altijd alleen "healthy" (de meest voorkomende combinatie, 4.346 van 16.816 trainingsbeelden). Op de validatieset haalt het een match ratio van 0.41, wat hoog lijkt maar misleidend is. 41% van de beelden heeft toevallig precies die combinatie. De macro F1 is maar 0.21, omdat "compromised", "dead" en "rubble" allemaal een F1 van 0.00 krijgen. Dit laat zien dat macro F1 de eerlijkere evaluatiemaat is voor deze taak [5].

## 10. Machine learning baseline

Hier wordt logistic regression op kleur en textuur gebruikt. Naast kleurhistogrammen gaan ook het gemiddelde en de standaardafwijking per kleurkanaal mee [6]. Zo wordt zichtbaar wat er mogelijk is met simpele kenmerken zonder deep learning.


In [ ]:
def extract_features(path, bins=32):
    arr = np.array(Image.open(path).convert('RGB').resize((128, 128)))
    feats = []
    for ch in range(3):
        h, _ = np.histogram(arr[:, :, ch], bins=bins, range=(0, 256))
        feats.extend(h / h.sum())
        feats.append(arr[:, :, ch].mean() / 255.0)
        feats.append(arr[:, :, ch].std() / 255.0)
    return feats

def predict_lr_baseline(models_per_label, X):
    preds = np.zeros((len(X), len(available_labels)), dtype=int)
    for i, lbl in enumerate(available_labels):
        model = models_per_label[lbl]
        if isinstance(model, dict) and model.get("type") == "constant":
            preds[:, i] = model["value"]
        else:
            preds[:, i] = model.predict(X)
    return preds

from concurrent.futures import ThreadPoolExecutor

def extract_features_parallel(filepaths, desc="Features berekenen"):
    with ThreadPoolExecutor(max_workers=NUM_WORKERS or 4) as ex:
        results = list(tqdm(ex.map(extract_features, filepaths), total=len(filepaths), desc=desc))
    return np.array(results)

print("Histogrammen berekenen...")
X_tr = extract_features_parallel(df_train['filepath'].tolist(), desc="Train")
X_vl = extract_features_parallel(df_val['filepath'].tolist(), desc="Val")

y_tr = df_train[available_labels].values
y_vl = df_val[available_labels].values
y_te = df_test[available_labels].values

lr_models = {}
for i, lbl in enumerate(available_labels):
    unique_classes = np.unique(y_tr[:, i])
    if len(unique_classes) == 1:
        lr_models[lbl] = {"type": "constant", "value": int(unique_classes[0])}
    else:
        lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
        lr.fit(X_tr, y_tr[:, i])
        lr_models[lbl] = lr

lr_val_preds = predict_lr_baseline(lr_models, X_vl)
lr_val_result = evaluate_hard_predictions(y_vl, lr_val_preds, available_labels)

results_val['LR baseline'] = {
    'macro_f1': lr_val_result['macro_f1'],
    'match_ratio': lr_val_result['match_ratio'],
    'set': 'val',
}
print_results(lr_val_result, available_labels, "Logistic regression baseline (validatie)")

### Bespreking van de logistic regression baseline

De logistic regression haalt een macro F1 van 0.44, meer dan het dubbele van het null model (0.21). Alle vier labels krijgen een F1 boven nul. "Healthy" scoort het hoogst (F1=0.68), "rubble" het laagst (F1=0.19). De match ratio (0.15) is lager dan het null model (0.41). De LR baseline probeert alle labels te voorspellen en maakt daarbij vaker fouten op beeldniveau, terwijl het null model altijd dezelfde combinatie raadt die het vaakst voorkomt.

## 11. CNN from scratch

Shao et al. gebruikten alleen modellen met ImageNet pretraining [5, 16]. Daarom wordt hier ook een CNN from scratch getraind, zodat zichtbaar is wat er mogelijk is zonder voorkennis.

Het netwerk heeft vijf convolutieblokken. Elk blok bestaat uit twee conv-lagen, batch normalization, ReLU, max pooling en dropout van 0.25. Het vijfde blok loopt op naar 512 filters. Na global average pooling volgt een fully connected laag met dropout 0.5. Voor een input van 512 bij 512 levert dit na vijf halveringen een feature map van 16 bij 16 op. Twee conv-lagen per blok geven betere feature extractie dan één [11].

Voor training from scratch worden lr=1e-3 en weight_decay=0.01 gebruikt als startpunt. De keuze voor lr=1e-3 is gebaseerd op de PyTorch implementatiedefault voor AdamW [22] en dient als startpunt voor training from scratch. Loshchilov en Hutter tonen aan dat gedecoupelde weight decay de generalisatie verbetert ten opzichte van L2-regularisatie [21]. Een learning rate van 1e-4 is een fine-tuning waarde en convergeert te langzaam bij een random netwerk. Een weight decay van 5e-4 is afkomstig uit SGD-literatuur en te laag voor AdamW's gedecoupelde weight decay formule [21]. De input wordt genormaliseerd met dataset-specifieke statistieken berekend op de trainingsset. Dit verschilt van de TL-modellen, waarvoor het aanbevolen is de bij de gekozen weights behorende preprocessing te volgen, omdat de pretrained gewichten op die verdeling zijn geoptimaliseerd [30]. De CNN from scratch heeft geen zulke verwachtingen en profiteert van normalisatie die aansluit bij de werkelijke beeldverdeling [30].

In [ ]:
class CoralCNN(nn.Module):
    def __init__(self, n):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Dropout2d(0.25),
            )

        self.block1 = conv_block(3, 32)
        self.block2 = conv_block(32, 64)
        self.block3 = conv_block(64, 128)
        self.block4 = conv_block(128, 256)
        self.block5 = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, n),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        return self.head(x)


def build_cnn_model(_arch_name, n):
    return CoralCNN(n).to(DEVICE)


set_seed()
num_labels = len(available_labels)
cnn_batch_size = estimate_train_batch_size(build_cnn_model, "cnn", num_labels, start_batch=ARCH_START_BATCH["cnn"])
arch_batch_sizes = {"cnn": cnn_batch_size}

train_loader, val_loader, _, _ = make_loaders(cnn_batch_size, eval_batch_size=min(cnn_batch_size * 2, 128), img_size=IMG_SIZE, mean=DATASET_MEAN, std=DATASET_STD)
print(f"Train loader batch size CNN: {cnn_batch_size}")

set_seed()
cnn = build_cnn_model("cnn", num_labels)
criterion = make_criterion()
optimizer = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print(f"Parameters: {sum(p.numel() for p in cnn.parameters()):,}")

cnn_ckpt = os.path.join(BASE_DIR, "cnn_scratch.pt")
cnn, cnn_h = train_model(
    cnn,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    available_labels,
    save_path=cnn_ckpt,
)
plot_curves(cnn_h, "CNN from scratch")
cnn_r = evaluate(cnn, val_loader, criterion, available_labels)
print_results(cnn_r, available_labels, "CNN from scratch (validatie)")
results_val["CNN from scratch"] = {
    "macro_f1": cnn_r["macro_f1"],
    "match_ratio": cnn_r["match_ratio"],
    "set": "val",
}
del cnn
clear_gpu()

### Bespreking van de CNN from scratch

De CNN met vijf convolutieblokken en dubbele conv-lagen per blok heeft meer capaciteit dan een ondiep netwerk voor een 512×512 input [11]. Het netwerk haalt een validatie macro F1 van 0.47 en een match ratio van 0.15. Dat is een duidelijke verbetering ten opzichte van de LR baseline (0.44), maar het trainingsverschil loopt op naarmate de epochs vorderen (train F1 stijgt tot 0.54, val F1 stabiliseert rond 0.47), wat wijst op lichte overfitting. De augmentatie-ablation hieronder test of dataaugmentatie dit gat kan verkleinen.


### 11.1 Augmentatie-ablation CNN from scratch

Augmentatie is voor een CNN from scratch relevanter dan voor fine-tuning. Onder pretrained features is het model kwetsbaarder voor overfitting op herhalende patronen in de trainingsset [8]. Vier transformaties worden gebruikt. Dat zijn horizontale en verticale flip, ColorJitter en GaussianBlur. De overige instellingen zijn identiek aan de hoofdtraining in §11, zodat alleen het effect van augmentatie zichtbaar is.

In [ ]:
print("CNN from scratch met augmentatie")
set_seed()
train_loader_aug, val_loader_aug, _, _ = make_loaders(
    arch_batch_sizes["cnn"],
    eval_batch_size=min(arch_batch_sizes["cnn"] * 2, 128),
    img_size=IMG_SIZE,
    aug=True,
    mean=DATASET_MEAN,
    std=DATASET_STD,
)

cnn_aug = build_cnn_model("cnn", num_labels)
criterion_aug = make_criterion()
optimizer_aug = optim.AdamW(cnn_aug.parameters(), lr=1e-3, weight_decay=0.01)
scheduler_aug = optim.lr_scheduler.CosineAnnealingLR(optimizer_aug, T_max=NUM_EPOCHS, eta_min=1e-6)

cnn_aug_ckpt = os.path.join(BASE_DIR, "cnn_scratch_aug.pt")
cnn_aug, cnn_aug_h = train_model(
    cnn_aug, train_loader_aug, val_loader_aug,
    criterion_aug, optimizer_aug, scheduler_aug,
    available_labels, save_path=cnn_aug_ckpt,
)
plot_curves(cnn_aug_h, "CNN from scratch met augmentatie")
cnn_aug_r = evaluate(cnn_aug, val_loader_aug, criterion_aug, available_labels)
print_results(cnn_aug_r, available_labels, "CNN from scratch + aug (validatie)")

aug_cnn_df = pd.DataFrame({
    "CNN (geen aug)": {"macro_f1": cnn_r["macro_f1"], "match_ratio": cnn_r["match_ratio"]},
    "CNN (aug)": {"macro_f1": cnn_aug_r["macro_f1"], "match_ratio": cnn_aug_r["match_ratio"]},
}).T
print("\nAugmentatie-effect op CNN from scratch (validatie)")
print(aug_cnn_df.to_string(float_format=lambda x: f"{x:.4f}"))

best_cnn_r = cnn_aug_r if cnn_aug_r["macro_f1"] > cnn_r["macro_f1"] else cnn_r
best_cnn_ckpt = cnn_aug_ckpt if cnn_aug_r["macro_f1"] > cnn_r["macro_f1"] else cnn_ckpt
results_val["CNN from scratch"] = {
    "macro_f1": best_cnn_r["macro_f1"],
    "match_ratio": best_cnn_r["match_ratio"],
    "set": "val",
}
del cnn_aug
del train_loader_aug, val_loader_aug
clear_gpu()

### Bespreking van de CNN augmentatie-ablation

Met augmentatie stijgt de CNN van 0.4720 naar 0.5012 macro F1 op validatie. De grootste winst zit bij "rubble", dat van F1=0.11 naar F1=0.23 gaat. Dat past bij het idee dat augmentatie vooral zeldzame labels helpt, omdat elk beeld tijdens training in meerdere varianten wordt gezien [8]. De match ratio daalt licht (0.1480 → 0.1381). Het model is minder vaak helemaal correct op beeldniveau, maar haalt wel stabielere per-label resultaten. De CNN-met-augmentatie wordt meegenomen naar de finale evaluatie.


## 12. Vergelijking van transfer learning modellen met full fine-tuning

Bij transfer learning hergebruik je een model dat al getraind is op een grote dataset zoals ImageNet [8]. Er zijn twee manieren: alleen de classificatielaag trainen (frozen backbone) of het volledige model opnieuw trainen (full fine-tuning) [23]. In dit project wordt full fine-tuning gebruikt, omdat de domeinafstand tussen ImageNet en onderwaterfotografie groot is [4]. ImageNet bestaat uit bovenwater-foto's met andere kleurbalans en textuurpatronen dan koraalbeelden. Shao et al. bevestigen dat een frozen backbone de laagste prestaties geeft op de Koh Tao dataset [16]. Gómez-Ríos et al. laten zien dat fine-tuning van CNN-modellen op koraaldata consistent beter presteert dan frozen features [8]. Bij full fine-tuning krijgt de backbone lr=1e-5 zodat pretrained kenmerken bewaard blijven en de classificatielaag lr=1e-4 [23]. De weight decay is 0.001 voor de backbone en 0.01 voor de classificatielaag, als startpunt gebaseerd op de PyTorch standaardwaarden voor AdamW [22] en de decoupled weight decay van Loshchilov en Hutter [21]. Bovendien worden gradiënten geclipped op max_norm=1.0 om instabiliteit bij mixed precision training te voorkomen [28]. Als scheduler wordt CosineAnnealingLR gebruikt [29] met een lineaire warmup van twee epochs. In de praktijk wordt warmup toegepast om te voorkomen dat de willekeurig geïnitialiseerde classificatielaag in de eerste epochs de pretrained backbone te sterk verstoort.

De drie architecturen zijn elk om een specifieke reden gekozen. EfficientNet-B7 schaalt breedte, diepte en resolutie gelijktijdig op [10]. CoralNet, het meestgebruikte platform voor koraalbeeldanalyse, gebruikt EfficientNet-B0 als backbone [15]. EfficientNet-B7 is de grootste variant in dezelfde familie en wordt hier gekozen vanwege de hogere modelcapaciteit voor de complexere multi-label taak. Shao et al. haalden hiermee macro F1 80,34% [16]. ResNet-101 gebruikt residual connections [11]; Shao et al. haalden macro F1 75,08% [16]. DenseNet-201 verbindt elke laag met alle voorgaande lagen [12]; Shao et al. haalden macro F1 77,87% [16]. Alle drie zijn gekozen, omdat Shao et al. ze op dezelfde beelden hebben getest, wat aantoont dat de architecturen geschikt zijn voor dit beelddomein [5, 16]. Een directe vergelijking van scores is echter niet mogelijk. Shao et al. classificeren acht labels (vier condities en vier stressoren), terwijl dit project alleen de vier conditielabels gebruikt. Macro F1 op vier labels is een andere meting dan op acht labels. Voor ResNet-101 en DenseNet-201 wordt een extra Dropout(0.5) laag toegevoegd vóór de classificatielaag. Dropout is een bewezen regularisatietechniek die willekeurig eenheden uitschakelt tijdens training en daarmee overfitting tegengaat [27]. Bij fine-tuning van grote netwerken op relatief kleine datasets zoals deze dataset met circa 17.000 trainingsbeelden is dit bijzonder relevant, omdat de classificatielaag anders snel de trainingsfeatures onthoudt. EfficientNet-B7 heeft al Dropout(0.5) in zijn originele classifier, zodat alle drie modellen op dit punt consistent zijn.

De modellen draaien allemaal op 512×512 pixels voor een gecontroleerde architectuurvergelijking. EfficientNet-B7 is ontworpen voor 600×600, ResNet-101 en DenseNet-201 voor 224×224. Dit is een beperking.

In [ ]:
def build_model(arch, n):
    if arch == "efficientnet_b7":
        m = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
        backbone_params = list(m.features.parameters())
        head_params = list(m.classifier.parameters())
    elif arch == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        in_features = m.fc.in_features
        m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, n))
        backbone_params = [p for name, p in m.named_parameters() if "fc" not in name]
        head_params = list(m.fc.parameters())
    elif arch == "densenet201":
        m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        in_features = m.classifier.in_features
        m.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, n))
        backbone_params = list(m.features.parameters())
        head_params = list(m.classifier.parameters())
    else:
        raise ValueError(f"Onbekende architectuur: {arch}")
    return m.to(DEVICE), backbone_params, head_params


checkpoints = {}
for arch in ["efficientnet_b7", "resnet101", "densenet201"]:
    print(f"Training {arch} (full fine-tuning)")

    set_seed()
    if arch not in arch_batch_sizes:
        def _build_for_bs(a, n): return build_model(a, n)[0]
        arch_batch_sizes[arch] = estimate_train_batch_size(_build_for_bs, arch, num_labels, start_batch=ARCH_START_BATCH[arch])

    train_bs = arch_batch_sizes[arch]
    eval_bs = min(train_bs * 2, 128)
    train_loader, val_loader, _, _ = make_loaders(train_bs, eval_bs, img_size=IMG_SIZE)
    print(f"Batch size train={train_bs}, eval={eval_bs}")
    set_seed()
    model, backbone_params, head_params = build_model(arch, num_labels)
    criterion = make_criterion()
    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": 1e-5, "weight_decay": 0.001},
        {"params": head_params,     "lr": 1e-4, "weight_decay": 0.01},
    ])
    warmup_epochs = 2
    warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-7)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])



    ckpt_path = os.path.join(BASE_DIR, f"{arch}_finetuned.pt")
    model, hist = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        available_labels, n_epochs=NUM_EPOCHS, patience=PATIENCE, save_path=ckpt_path,
    )
    plot_curves(hist, f"{arch} (full fine-tuning)")
    val_result = evaluate(model, val_loader, criterion, available_labels)
    print_results(val_result, available_labels, f"{arch} validatie")
    results_val[arch] = {
        "macro_f1": val_result["macro_f1"],
        "match_ratio": val_result["match_ratio"],
        "set": "val",
    }
    checkpoints[arch] = ckpt_path
    del model
    clear_gpu()

In [ ]:
val_selection_df = (
    pd.DataFrame(results_val)
    .T[["match_ratio", "macro_f1", "set"]]
    .sort_values("macro_f1", ascending=False)
)
print(val_selection_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

transfer_candidates = [k for k in ["efficientnet_b7", "resnet101", "densenet201"] if k in results_val]
best_arch = max(transfer_candidates, key=lambda k: results_val[k]["macro_f1"])
print(f"Beste transfermodel tot nu toe: {best_arch}")

### Bespreking van de transfermodellen

De drie transfer learning modellen halen op validatie: EfficientNet-B7 macro F1 = 0.6209, DenseNet-201 = 0.6171 en ResNet-101 = 0.5903. Alle drie verslaan de CNN from scratch (0.5012) en de LR baseline (0.4402) met ruime marge, wat bevestigt dat ImageNet-features ondanks de domeinafstand naar onderwaterbeelden nuttig blijven [16, 8]. EfficientNet-B7 en DenseNet-201 liggen dicht bij elkaar. EfficientNet-B7 wordt als winnaar meegenomen naar de ablations in §13 op basis van de hoogste validatie macro F1.


## 13. Gerichte ablations

Op het best presterende model worden drie factoren apart getest. Per vergelijking wordt één factor gevarieerd terwijl de rest vastgehouden wordt. De volgorde is dat de beste waarde uit elke stap wordt meegenomen naar de volgende. De drie vergelijkingen zijn: learning rate (13.1), beeldresolutie (13.2) en augmentatie (13.3).

In [ ]:
def build_model_f(arch, n):
    if arch == "efficientnet_b7":
        m = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
        backbone_params = list(m.features.parameters())
        head_params = list(m.classifier.parameters())
    elif arch == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        in_features = m.fc.in_features
        m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, n))
        backbone_params = [p for name, p in m.named_parameters() if "fc" not in name]
        head_params = list(m.fc.parameters())
    elif arch == "densenet201":
        m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        in_features = m.classifier.in_features
        m.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, n))
        backbone_params = list(m.features.parameters())
        head_params = list(m.classifier.parameters())
    else:
        raise ValueError(f"Onbekende architectuur: {arch}")
    return m.to(DEVICE), backbone_params, head_params


def run_ablation_experiment(arch, n, lr_head, res, tag, aug=False, seed=None):
    # Als een seed expliciet wordt meegegeven (bv. voor de robuustheidstest),
    # gebruik die. Anders default SEED voor reproduceerbare standaardruns.
    set_seed(seed if seed is not None else SEED)

    def _build_for_bs(a, n_out): return build_model_f(a, n_out)[0]

    batch_key = f"{arch}_{res}_ft"
    if batch_key not in arch_batch_sizes:
        start_batch = ARCH_START_BATCH.get(arch, 8)
        if res == 224:
            start_batch = min(start_batch * 2, 128)
        arch_batch_sizes[batch_key] = estimate_train_batch_size(
            _build_for_bs, arch, n, start_batch=start_batch, img_size=res,
        )

    train_bs = arch_batch_sizes[batch_key]
    eval_bs = min(train_bs * 2, 128)
    train_loader, val_loader, _, _ = make_loaders(train_bs, eval_bs, img_size=res, aug=aug)

    set_seed(seed if seed is not None else SEED)
    model, backbone_params, head_params = build_model_f(arch, n)
    criterion = make_criterion()
    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": 1e-5,     "weight_decay": 0.001},
        {"params": head_params,     "lr": lr_head,  "weight_decay": 0.01},
    ])
    warmup_epochs = 2
    warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-7)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])

    ckpt = os.path.join(BASE_DIR, f"{tag}.pt")
    model, hist = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        available_labels, n_epochs=NUM_EPOCHS, patience=PATIENCE, save_path=ckpt,
    )
    val_result = evaluate(model, val_loader, criterion, available_labels, threshold=0.5)
    del model
    clear_gpu()

    return {
        "lr_head": lr_head, "res": res, "aug": aug,
        "checkpoint": ckpt, "history": hist, "val_result": val_result,
        "train_bs": train_bs, "eval_bs": eval_bs,
    }

### 13.1 Learning rate vergelijking

De eerste ablation vergelijkt lr_head=1e-4 (uit Shao et al. [16]) met lr_head=1e-3 (de implementatiedefault van PyTorch AdamW [22]). De backbone learning rate blijft 1e-5 en de resolutie is 512 pixels. Zo is alleen het effect van de head learning rate zichtbaar.

In [ ]:
lr_study = {}
lr_configs = [
    {"name": "lr_1e4", "lr": 1e-4, "res": 512},
    {"name": "lr_1e3", "lr": 1e-3, "res": 512},
]

print(f"Learning-rate studie op {best_arch} met full fine-tuning en resolutie 512")
for cfg in lr_configs:
    print(f"Run: {cfg['name']}")
    out = run_ablation_experiment(
        best_arch, num_labels,
        lr_head=cfg["lr"], res=cfg["res"],
        tag=f"lr_study_{best_arch}_{cfg['name']}",
        aug=False,
    )
    lr_study[cfg["name"]] = out
    plot_curves(out["history"], f"LR studie: {cfg['name']}")

lr_study_df = pd.DataFrame({
    name: {
        "macro_f1": out["val_result"]["macro_f1"],
        "match_ratio": out["val_result"]["match_ratio"],
        "lr_head": out["lr_head"],
        "res": out["res"],
    }
    for name, out in lr_study.items()
}).T.sort_values("macro_f1", ascending=False)

print("\nResultaten learning-rate studie op validatie")
print(lr_study_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

best_lr_name = lr_study_df.index[0]
best_lr = lr_study[best_lr_name]["lr_head"]
print(f"Beste learning rate: {best_lr} uit run {best_lr_name}")

### Bespreking van de learning rate vergelijking

lr_head = 1e-4 haalt 0.6209 en lr_head = 1e-3 haalt 0.6253 op validatie. De winst is klein, maar consistent met de aanbeveling uit de AdamW-literatuur dat de classificatiekop een hogere learning rate mag hebben dan de pretrained backbone [21]. De hogere learning rate leidt wel tot sterkere overfitting in latere epochs (train F1 loopt op tot 0.90 terwijl val stabiliseert rond 0.62), maar dankzij early stopping op validatie-F1 heeft dat geen invloed op het gekozen checkpoint. lr_head = 1e-3 wordt meegenomen naar de volgende ablations.


### 13.2 Resolutie vergelijking

De tweede ablation vergelijkt 512 pixels (de originele patchgrootte) met 224 pixels. De learning rate en backbone status worden vastgehouden op de beste waarde uit 13.1.

In [ ]:
res_study = {}
res_configs = [
    {"name": "res_512", "lr": best_lr, "res": 512},
    {"name": "res_224", "lr": best_lr, "res": 224},
]

print(f"\nResolutiestudie op {best_arch} met full fine-tuning en learning rate {best_lr}")
for cfg in res_configs:
    print(f"Run: {cfg['name']}")
    out = run_ablation_experiment(
        best_arch, num_labels,
        lr_head=cfg["lr"], res=cfg["res"],
        tag=f"res_study_{best_arch}_{cfg['name']}",
        aug=False,
    )
    res_study[cfg["name"]] = out
    plot_curves(out["history"], f"Resolutie studie: {cfg['name']}")

res_study_df = pd.DataFrame({
    name: {
        "macro_f1": out["val_result"]["macro_f1"],
        "match_ratio": out["val_result"]["match_ratio"],
        "lr_head": out["lr_head"],
        "res": out["res"],
    }
    for name, out in res_study.items()
}).T.sort_values("macro_f1", ascending=False)

print("\nResultaten resolutiestudie op validatie")
print(res_study_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

best_res_name = res_study_df.index[0]
best_res = int(res_study[best_res_name]["res"])
print(f"Beste resolutie: {best_res} uit run {best_res_name}")

### Bespreking van de resolutie vergelijking

512 pixels haalt 0.6253 macro F1 op validatie, 224 pixels haalt 0.6108. De originele patchgrootte uit de Koh Tao dataset is 512 [5], dus verkleinen naar 224 gooit bruikbare textuurinformatie weg. 512 pixels wordt meegenomen naar de augmentatie-ablation en de uiteindelijke setting.


### 13.3 Augmentatie vergelijking

De derde ablation vergelijkt training zonder augmentatie versus met augmentatie. Augmentatie omvat vier transformaties die typisch zijn voor onderwaterbeelden [8]:

- `RandomHorizontalFlip(p=0.5)` en `RandomVerticalFlip(p=0.5)`: koraal heeft geen vaste oriëntatie.
- `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)`: simuleert variabele waterdiepte en lichtinval. Hue wordt laag gehouden (0.05) om de conditieinformatie in het koraal niet te vervalsen.
- `GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))`: simuleert lichte watertroebelheid.

De learning rate en resolutie worden vastgehouden op de beste waarden uit 13.1 en 13.2.

In [ ]:
aug_study = {}
aug_configs = [
    {"name": "aug_off", "aug": False},
    {"name": "aug_on",  "aug": True},
]

print(f"Augmentatiestudie op {best_arch} met full fine-tuning, lr={best_lr}, res={best_res}")
for cfg in aug_configs:
    print(f"Run: {cfg['name']}")
    out = run_ablation_experiment(
        best_arch, num_labels,
        lr_head=best_lr, res=best_res,
        tag=f"aug_study_{best_arch}_{cfg['name']}",
        aug=cfg["aug"],
    )
    aug_study[cfg["name"]] = out
    plot_curves(out["history"], f"Augmentatie studie: {cfg['name']}")

aug_study_df = pd.DataFrame({
    name: {
        "macro_f1": out["val_result"]["macro_f1"],
        "match_ratio": out["val_result"]["match_ratio"],
        "aug": out["aug"],
    }
    for name, out in aug_study.items()
}).T.sort_values("macro_f1", ascending=False)

print("\nResultaten augmentatiestudie op validatie")
print(aug_study_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

best_aug_name = aug_study_df.index[0]
best_aug = aug_study[best_aug_name]["aug"]
best_final_ckpt = aug_study[best_aug_name]["checkpoint"]
best_setting_name = f"{best_arch}_lr={best_lr}_res={best_res}_aug={best_aug}"
print(f"Beste instelling: {best_setting_name}")

### Bespreking van de augmentatie vergelijking

Augmentatie geeft op EfficientNet-B7 de grootste winst van de drie ablations: 0.6253 zonder augmentatie naar 0.6533 met augmentatie op validatie. Dit past bij de bevinding van Gomez-Rios et al. dat augmentatie bij koraalclassificatie helpt, vooral onder beperkte dataomvang en bij onderwaterbeelden waar kleur en belichting variëren tussen surveys [8]. De winst is groter dan bij de CNN from scratch, terwijl de absolute score veel hoger ligt. Augmentatie helpt dus bij beide model-familieën op ongeveer dezelfde schaal. De beste instelling wordt: EfficientNet-B7 met lr_head = 1e-3, resolutie 512 en augmentatie aan.


### 13.4 Robuustheid met meerdere seeds

Een enkel trainingsresultaat is vatbaar voor willekeur in de training. Daarom wordt zowel de CNN from scratch als het beste transfermodel drie keer getraind met verschillende seeds. De gemiddelde macro F1 en de standaarddeviatie geven een betrouwbaarder beeld van de werkelijke prestaties [14].

In [ ]:
seeds_to_test = [42, 123, 456]

cnn_seed_results = []
print("CNN from scratch robuustheidstest")
for s in seeds_to_test:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

    train_loader, val_loader, _, _ = make_loaders(
        arch_batch_sizes["cnn"], min(arch_batch_sizes["cnn"] * 2, 128), img_size=IMG_SIZE,
        mean=DATASET_MEAN, std=DATASET_STD
    )
    m = build_cnn_model("cnn", num_labels)
    crit = make_criterion()
    opt = optim.AdamW(m.parameters(), lr=1e-3, weight_decay=0.01)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=1e-6)
    ckpt_s = os.path.join(BASE_DIR, f"cnn_seed{s}.pt")
    m, _ = train_model(m, train_loader, val_loader, crit, opt, sch, available_labels, save_path=ckpt_s)
    r = evaluate(m, val_loader, crit, available_labels)
    cnn_seed_results.append(r["macro_f1"])
    print(f"  seed={s}  macro_f1={r['macro_f1']:.4f}")
    del m; clear_gpu()

tl_seed_results = []
print(f"\n{best_arch} robuustheidstest (beste instelling)")
for s in seeds_to_test:
    out = run_ablation_experiment(
        best_arch, num_labels,
        lr_head=best_lr, res=best_res,
        tag=f"tl_seed{s}",
        aug=best_aug,
        seed=s,
    )
    tl_seed_results.append(out["val_result"]["macro_f1"])
    print(f"  seed={s}  macro_f1={out['val_result']['macro_f1']:.4f}")


print(f"\nCNN  macro F1: {np.mean(cnn_seed_results):.4f} ± {np.std(cnn_seed_results):.4f}")
print(f"TL   macro F1: {np.mean(tl_seed_results):.4f} ± {np.std(tl_seed_results):.4f}")

### Bespreking van de robuustheidstest

Een lage standaarddeviatie bevestigt dat de resultaten niet van één toevallige initialisatie afhangen [14]. De intervallen (gemiddelde ± standaarddeviatie) van de twee modellen overlappen elkaar niet. De CNN heeft een gemiddelde macro F1 van 0.4825 ± 0.0038 en de transfer learning gemiddelde macro F1: 0.6533 ± 0.0000. Dus de modellen zijn robuust.


## 14. Drempeloptimalisatie per label

Een vaste drempel van 0.5 is niet altijd de beste keuze voor elk label. Daarom wordt hier per label de drempel gezocht die de F1 score op de validatieset het hoogst maakt. Daarna worden die gekozen drempels één keer op de testset gebruikt. Het model dat geladen wordt is het beste model uit de augmentatie ablation (13.3).

In [ ]:
criterion = make_criterion()

best_model, _, _ = build_model_f(best_arch, num_labels)
best_model.load_state_dict(torch.load(best_final_ckpt, map_location=DEVICE, weights_only=True))
best_model = best_model.to(DEVICE)

best_batch_key = f"{best_arch}_{best_res}_ft"
best_train_bs = arch_batch_sizes[best_batch_key]
best_eval_bs = min(best_train_bs * 2, 128)

train_loader, val_loader, test_loader, best_eval_transform = make_loaders(
    best_train_bs,
    eval_batch_size=best_eval_bs,
    img_size=best_res,
    aug=False,
)

val_default = evaluate(best_model, val_loader, criterion, available_labels, threshold=0.5)
best_thresholds, thr_df = tune_thresholds_per_label(
    val_default["all_targets"],
    val_default["all_probs"],
    available_labels,
)
val_tuned = evaluate(best_model, val_loader, criterion, available_labels, threshold=best_thresholds)

print("Geoptimaliseerde drempels per label")
print(thr_df.to_string(index=False))

print_results(val_default, available_labels, "Winnend model op validatie met drempel 0.50")
print_results(val_tuned, available_labels, "Winnend model op validatie met geoptimaliseerde drempels")

### Bespreking van de drempeloptimalisatie

De geoptimaliseerde drempels zijn: healthy = 0.60, compromised = 0.55, dead = 0.50, rubble = 0.50. In dit project krijgt juist het meest voorkomende label ("healthy") de hoogste drempel, omdat het model daar een zeer hoge recall (0.94) bij een lagere precision haalde. Een strengere drempel tilt precision op zonder recall veel te schaden. De zeldzame labels "dead" en "rubble" blijven op 0.50, omdat hun kansen al laag zijn. Optimalisatie tilt validatie macro F1 van 0.6533 naar 0.6570. Dat het per label verschilt bevestigt dat één vaste drempel niet optimaal is bij onbalans [18].


## 15. Finale evaluatie op de testset

Het gekozen model wordt één keer op de testset geëvalueerd. Dat gebeurt met de standaarddrempel en ook met de geoptimaliseerde drempels. Voor een eerlijke vergelijking gaan het null model en de logistic regression baseline en de CNN from scratch ook op precies dezelfde testset.


In [ ]:
X_te = extract_features_parallel(df_test['filepath'].tolist(), desc="Test")
y_te = df_test[available_labels].values

In [ ]:
test_default = evaluate(best_model, test_loader, criterion, available_labels, threshold=0.5)
test_tuned = evaluate(best_model, test_loader, criterion, available_labels, threshold=best_thresholds)

print_results(test_default, available_labels, "Beste model op test met drempel 0.50")
print_results(test_tuned, available_labels, f"Beste model op test met gekozen drempels {list(np.round(best_thresholds, 2))}")

# Baselines op dezelfde testset
null_test_preds = np.tile(null_pred, (len(y_te), 1))
null_test_result = evaluate_hard_predictions(y_te, null_test_preds, available_labels)

lr_test_preds = predict_lr_baseline(lr_models, X_te)
lr_test_result = evaluate_hard_predictions(y_te, lr_test_preds, available_labels)

cnn_test_loader = make_loaders(
    arch_batch_sizes["cnn"],
    eval_batch_size=min(arch_batch_sizes["cnn"] * 2, 128),
    img_size=IMG_SIZE,
    mean=DATASET_MEAN,
    std=DATASET_STD,
)[2]
cnn_model = build_cnn_model("cnn", num_labels)
cnn_model.load_state_dict(torch.load(best_cnn_ckpt, map_location=DEVICE, weights_only=True))
cnn_model = cnn_model.to(DEVICE)
cnn_test_result = evaluate(cnn_model, cnn_test_loader, make_criterion(), available_labels, threshold=0.5)
del cnn_model
clear_gpu()

results_test["Null model"] = {
    "macro_f1": null_test_result["macro_f1"],
        "match_ratio": null_test_result["match_ratio"],
    "set": "test",
}
results_test["LR baseline"] = {
    "macro_f1": lr_test_result["macro_f1"],
        "match_ratio": lr_test_result["match_ratio"],
    "set": "test",
}
results_test["CNN from scratch"] = {
    "macro_f1": cnn_test_result["macro_f1"],
        "match_ratio": cnn_test_result["match_ratio"],
    "set": "test",
}
results_test[f"Beste model ({best_setting_name}, 0.5)"] = {
    "macro_f1": test_default["macro_f1"],
    "match_ratio": test_default["match_ratio"],
    "set": "test",
}
results_test[f"Beste model ({best_setting_name}, tuned)"] = {
    "macro_f1": test_tuned["macro_f1"],
    "match_ratio": test_tuned["match_ratio"],
    "set": "test",
}

test_comparison_df = (
    pd.DataFrame(results_test)
    .T[["match_ratio", "macro_f1", "set"]]
    .sort_values("macro_f1", ascending=False)
)
print("\nEerlijke vergelijking op dezelfde testset")
print(test_comparison_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))


### Bespreking van de finale testresultaten

Op de testset haalt het winnende model macro F1 = 0.6604 met drempel 0.50 en 0.6595 met de geoptimaliseerde drempels. De drempeloptimalisatie verbetert validatie-F1 maar op test is het verschil vrijwel nul en licht negatief, wat erop wijst dat de drempels licht overfit zijn op de validatieverdeling [18]. De match ratio verbetert wel: van 0.3515 naar 0.3789. Alle modellen zijn op dezelfde testset vergeleken: null 0.2120, LR baseline 0.4557, CNN 0.5123, winnend transfermodel 0.6595.


### Verificatie van modelvereisten

Hieronder wordt automatisch gecontroleerd of het eindmodel aan alle zes vereisten voldoet.

In [ ]:
null_f1_test = null_test_result["macro_f1"]
lr_f1_test   = lr_test_result["macro_f1"]
best_f1_test = test_tuned["macro_f1"]
per_label_f1 = [test_tuned["per_label"][l]["f1"] for l in available_labels]

# Vereiste 4: overfitting check op het beste checkpoint (hoogste val F1)
best_hist = aug_study[best_aug_name]["history"]
train_f1_key = next((k for k in ["train_f1", "train_macro_f1", "macro_f1_train"] if k in best_hist), None)
val_f1_key   = next((k for k in ["val_f1", "val_macro_f1", "macro_f1_val"] if k in best_hist), None)
if train_f1_key and val_f1_key and best_hist[train_f1_key] and best_hist[val_f1_key]:
    best_epoch = int(np.argmax(best_hist[val_f1_key]))
    last_train_f1 = best_hist[train_f1_key][best_epoch]
    last_val_f1   = best_hist[val_f1_key][best_epoch]
    overfit_gap   = abs(last_train_f1 - last_val_f1)
else:
    print("Let op: train_f1 niet gevonden in history. Vereiste 4 kan niet automatisch gecontroleerd worden.")
    last_train_f1 = float("nan")
    last_val_f1   = float("nan")
    overfit_gap   = float("nan")

# Vereiste 5: robuustheid over seeds
cnn_std = float(np.std(cnn_seed_results))
tl_std  = float(np.std(tl_seed_results))

# Vereiste 6: drempeloptimalisatie verbetert validatie F1
val_f1_default = val_default["macro_f1"]
val_f1_tuned   = val_tuned["macro_f1"]

checks = {
    "1. Macro F1 test > null model":         best_f1_test > null_f1_test,
    "2. Macro F1 test > LR baseline":        best_f1_test > lr_f1_test,
    "3. Alle labels F1 > 0.00":              all(f > 0.0 for f in per_label_f1),
    "4. Overfit gap (train-val F1) <= 0.20": overfit_gap <= 0.20 if math.isfinite(overfit_gap) else None,
    "5a. CNN seed std < 0.05":               cnn_std < 0.05,
    "5b. TL seed std < 0.05":                tl_std  < 0.05,
    "6. Drempels verbeteren val F1":         val_f1_tuned > val_f1_default,
}

print("Verificatie modelvereisten")
all_pass = True
for naam, result in checks.items():
    status = "NIET GECONTROLEERD" if result is None else ("PASS" if result else "FAIL")
    if result is None or not result:
        all_pass = False
    print(f"  [{status}] {naam}")

print()
print(f"Null model macro F1:       {null_f1_test:.4f}")
print(f"LR baseline macro F1:      {lr_f1_test:.4f}")
print(f"Beste model macro F1:      {best_f1_test:.4f}")
print(f"Per-label F1:              {[round(f,3) for f in per_label_f1]}")
print(f"Overfit gap (train-val):   {overfit_gap:.4f}")
print(f"CNN seed std:              {cnn_std:.4f}")
print(f"TL seed std:               {tl_std:.4f}")
print(f"Val F1 default drempel:    {val_f1_default:.4f}")
print(f"Val F1 geoptimaliseerde:   {val_f1_tuned:.4f}")
print()
if all_pass:
    print("Alle vereisten gehaald.")
else:
    print("Niet alle vereisten aangetoond. Zie FAIL of NIET GECONTROLEERD hierboven.")

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
x = np.arange(len(available_labels)); w = 0.25
ax.bar(x-w, [test_tuned['per_label'][l]['precision'] for l in available_labels], w, label='Precision', color='#3498db')
ax.bar(x,   [test_tuned['per_label'][l]['recall'] for l in available_labels], w, label='Recall', color='#e74c3c')
ax.bar(x+w, [test_tuned['per_label'][l]['f1'] for l in available_labels], w, label='F1', color='#2ecc71')
ax.set_xticks(x); ax.set_xticklabels(available_labels)
ax.set_ylabel('Score'); ax.set_title('Per-label prestaties (test, geoptimaliseerde drempels)')
ax.legend(); ax.set_ylim(0,1.05); plt.tight_layout(); plt.show()

### FN en FP analyse per label

Shao et al. keken ook naar de verhouding tussen false negatives en false positives per label [5]. Een hoge FN/FP verhouding betekent dat het model een conditie vaak mist. Een lage verhouding onder 1 betekent dat het model juist te vaak alarm slaat. Deze analyse geeft extra inzicht naast de F1 scores per label.


In [ ]:
# FN/FP analyse per label (vergelijk met Shao et al. Table 2)
final_preds = test_tuned['all_preds']
final_targets = test_tuned['all_targets']

print('FN en FP per label op de testset met gekozen drempels')
print(f'{"Label":<15} {"FN":>6} {"FP":>6} {"FN/FP":>8}')
for i, lbl in enumerate(available_labels):
    fn = int(((final_targets[:, i] == 1) & (final_preds[:, i] == 0)).sum())
    fp = int(((final_targets[:, i] == 0) & (final_preds[:, i] == 1)).sum())
    ratio = fn / fp if fp > 0 else float('inf')
    print(f'{lbl:<15} {fn:>6} {fp:>6} {ratio:>8.2f}')


### Bespreking van false negatives en false positives

De FN/FP ratio per label op de testset is: healthy 0.44, compromised 0.69, dead 0.18, rubble 0.30. Alle ratios liggen onder 1, wat betekent dat het model voor elk label meer false positives dan false negatives produceert. Shao et al. rapporteren het tegenovergestelde patroon bij rubble (FN/FP = 1.90) [5]. Dit verschil is waarschijnlijk te verklaren door de combinatie van Asymmetric Loss die negatieve klassen zwaarder bestraft [25] en de geoptimaliseerde drempels die recall prioriteren. Voor een monitoringsysteem is dit gunstig: een gemiste aanwezigheid van "dead" of "rubble" is schadelijker dan een vals alarm.


### Confusion matrices per label

Voor elk conditielabel staat een aparte 2 bij 2 confusion matrix. Zo is snel te zien waar het model per label de mist in gaat. Het gaat dan vooral om vaak missen of juist te vaak vals alarm geven.


In [ ]:
from sklearn.metrics import confusion_matrix as sklearn_cm

fig, axes = plt.subplots(1, len(available_labels), figsize=(4*len(available_labels), 4))
for i, lbl in enumerate(available_labels):
    cm = sklearn_cm(final_targets[:, i], final_preds[:, i], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    axes[i].set_title(lbl)
    axes[i].set_xlabel('Voorspeld')
    axes[i].set_ylabel('Werkelijk')
plt.suptitle('Per-label confusion matrices (test, geoptimaliseerde drempels)', fontsize=13)
plt.tight_layout()
plt.show()

### Bespreking van de confusion matrices

De matrices hierboven tonen per label de verdeling van true positives, false positives, true negatives en false negatives. "Dead" en "rubble" lijken visueel op elkaar, wat kan leiden tot verwarring tussen deze labels [5].

## 15b. GradCAM analyse van misclassificaties

GradCAM laat zien waar het model naar kijkt [19]. In tegenstelling tot willekeurig gekozen voorbeelden richten we ons hier op misclassificaties: beelden waar het model een fout maakt. Zo wordt duidelijk welke visuele patronen het model misleiden.

Per label zoeken we een false negative (label aanwezig maar niet voorspeld). Als er geen false negative is, gebruiken we een false positive (label voorspeld maar niet aanwezig). Bij elk beeld worden de werkelijke labels (True) en de voorspelde labels (Pred) getoond.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._fwd_handle = target_layer.register_forward_hook(self._save_activation)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()
        if output.requires_grad:
            output.register_hook(self._save_gradient_tensor)

    def _save_gradient_tensor(self, grad):
        self.gradients = grad.detach()

    def generate(self, input_tensor, target_class):
        self.model.eval()
        self.gradients = None
        self.activations = None

        input_tensor = input_tensor.clone().detach().requires_grad_(True)
        output = self.model(input_tensor)
        self.model.zero_grad(set_to_none=True)

        score = output[0, target_class]
        score.backward(retain_graph=False)

        if self.gradients is None or self.activations is None:
            raise RuntimeError(
                "GradCAM kon geen gradiënten ophalen. Controleer of het model in eval mode staat "
                "en of de input requires_grad=True heeft."
            )

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam.squeeze().float().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


if "efficientnet" in best_arch:
    target_layer = best_model.features[-1]
elif "resnet" in best_arch:
    target_layer = best_model.layer4[-1]
elif "densenet" in best_arch:
    target_layer = best_model.features.denseblock4
else:
    raise ValueError(f"GradCAM niet ingericht voor: {best_arch}")

gradcam = GradCAM(best_model, target_layer)
best_model.eval()

all_targets_arr = test_tuned["all_targets"]
all_preds_arr = test_tuned["all_preds"]
df_test_reset = df_test.reset_index(drop=True)

fig, axes = plt.subplots(len(available_labels), 3, figsize=(12, 4 * len(available_labels)))
if len(available_labels) == 1:
    axes = np.array([axes])

for row, label in enumerate(available_labels):
    label_idx = available_labels.index(label)

    fn_indices = np.where(
        (all_targets_arr[:, label_idx] == 1) & (all_preds_arr[:, label_idx] == 0)
    )[0]
    fp_indices = np.where(
        (all_targets_arr[:, label_idx] == 0) & (all_preds_arr[:, label_idx] == 1)
    )[0]

    if len(fn_indices) > 0:
        chosen_idx = int(fn_indices[0])
        misclass_type = "FN"
    elif len(fp_indices) > 0:
        chosen_idx = int(fp_indices[0])
        misclass_type = "FP"
    else:
        for ax in axes[row]:
            ax.axis("off")
        axes[row][0].set_title(f"{label}: geen misclassificaties")
        continue

    sample = df_test_reset.iloc[chosen_idx]
    true_labels = [available_labels[j] for j in range(len(available_labels)) if all_targets_arr[chosen_idx, j] == 1]
    pred_labels = [available_labels[j] for j in range(len(available_labels)) if all_preds_arr[chosen_idx, j] == 1]

    img_pil = Image.open(sample["filepath"]).convert("RGB")
    img_tensor = best_eval_transform(img_pil).unsqueeze(0).to(DEVICE)
    if DEVICE.type == "cuda":
        img_tensor = img_tensor.contiguous(memory_format=torch.channels_last)

    cam = gradcam.generate(img_tensor, label_idx)
    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize(img_pil.size)
    )

    overlay = np.array(img_pil).copy().astype(float)
    heat = plt.cm.jet(cam_resized / 255.0)[..., :3] * 255
    blended = (0.5 * overlay + 0.5 * heat).clip(0, 255).astype(np.uint8)

    axes[row][0].imshow(img_pil)
    axes[row][0].set_title(f"{label} ({misclass_type})\nTrue: {true_labels}", fontsize=8)
    axes[row][0].axis("off")

    axes[row][1].imshow(cam_resized, cmap="jet")
    axes[row][1].set_title(f"GradCAM\nPred: {pred_labels}", fontsize=8)
    axes[row][1].axis("off")

    axes[row][2].imshow(blended)
    axes[row][2].set_title("Overlay", fontsize=8)
    axes[row][2].axis("off")

plt.suptitle("GradCAM op misclassificaties per label (FN = gemist, FP = fout positief)", fontsize=11)
plt.tight_layout()
plt.show()

### Bespreking van GradCAM

De GradCAM-heatmaps visualiseren welke regio's het sterkst bijdragen aan de voorspelling van het model [19]. Door ze te beperken tot misclassificaties wordt duidelijker waar de beslissingsgrens van het model zwak is dan bij correct geclassificeerde beelden. Bij de geïnspecteerde false negatives is de activatie vaak niet geconcentreerd op een duidelijke koraalregio, wat suggereert dat het model de bijbehorende textuur in dat specifieke beeld niet als signaal oppikt. Bij false positives zit de activatie op regio's die qua textuur of kleur lijken op het voorspelde label maar in werkelijkheid tot een ander label horen.

Dit sluit aan bij de per-label testresultaten: "compromised" heeft de laagste F1 (0.514) en vertoont visueel de grootste variatie binnen het label [5]. "Rubble" (F1 = 0.604) komt minder vaak voor in de trainingsset (1.850 beelden) en lijkt in textuur op dood koraal, wat verwarring tussen deze twee labels verklaart.

## 16. Samenvatting van de resultaten


In [ ]:
results_val[f"Beste transfermodel ({best_setting_name})"] = {
    "macro_f1": val_default["macro_f1"],
        "match_ratio": val_default["match_ratio"],
    "set": "val",
}
results_val[f"Beste transfermodel ({best_setting_name}, tuned)"] = {
    "macro_f1": val_tuned["macro_f1"],
        "match_ratio": val_tuned["match_ratio"],
    "set": "val",
}

val_report_df = (
    pd.DataFrame(results_val)
    .T[["match_ratio", "macro_f1", "set"]]
    .sort_values("macro_f1", ascending=False)
)

test_report_df = (
    pd.DataFrame(results_test)
    .T[["match_ratio", "macro_f1", "set"]]
    .sort_values("macro_f1", ascending=False)
)

print("Overzicht validatieresultaten")
print(val_report_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

print("\nOverzicht testresultaten")
print(test_report_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))


### Samenvatting

De overzichtstabel hierboven laat de consistente stijgende lijn zien: null model 0.21 → LR baseline 0.44 → CNN from scratch 0.51 → beste transfermodel 0.66. Op de testset haalt het winnende EfficientNet-B7 model (lr_head = 1e-3, resolutie 512, augmentatie aan) een macro F1 van 0.6604 met vaste drempel 0.50 en 0.6595 met per-label drempels. De drie ablations tonen aan dat augmentatie de grootste individuele bijdrage levert (+0.028), gevolgd door resolutie (+0.015) en learning rate (+0.004). De robuustheidstest bevestigt dat deze resultaten niet van één toevallige run afhangen.


## 17. Discussie

De opbouw van null model (0.21) naar LR baseline (0.44) naar CNN from scratch (0.51) naar transfer learning (0.66) laat bij elke stap een duidelijke winst zien op macro F1. De grootste sprong zit tussen CNN en transfer learning (+0.15), wat bevestigt dat ImageNet-features ondanks de domeinafstand naar onderwaterfotografie waardevol blijven [16, 8].

### Per-label patronen

"Healthy" is het makkelijkst te herkennen door de heldere kleuren en hoge prevalentie (F1 op test = 0.885) [5]. "Dead" en "rubble" scoren op test respectievelijk 0.635 en 0.604, zelfs hoger dan op validatie, doordat de testset relatief meer voorbeelden van deze labels bevat. "Compromised" heeft de laagste test-F1 (0.514), passend bij de observatie dat dit label visueel het meest gevarieerd is [5].

### Precision en recall tradeoff

Voor een vroegwaarschuwingssysteem is recall belangrijker dan precision. De FN/FP-ratio ligt voor alle labels onder 1, wat betekent dat het model eerder te veel dan te weinig waarschuwt. Voor monitoringdoeleinden is dat de voorkeursrichting.

### Gecontroleerde vergelijking van architecturen

De drie transfer learning modellen draaien allemaal op 512×512, niet op hun native resolutie. EfficientNet-B7 is ontworpen voor 600×600, ResNet-101 en DenseNet-201 voor 224×224. Dit maakt de vergelijking gecontroleerd maar niet volledig eerlijk per model. De resolutie-ablation (13.2) toont aan dat 512 voor EfficientNet-B7 beter werkt dan 224.

### Vergelijking met Shao et al.

De resultaten zijn niet direct vergelijkbaar met Shao et al. en mogen niet als benchmark gelezen worden. Shao et al. gebruiken acht labels, geavanceerdere architecturen zoals DINOv2 met LoRA-adapters [16] en een andere split-strategie.

### Beperkingen

De data komt uit een enkel gebied (Koh Tao) [5], wat de generaliseerbaarheid beperkt. De group-aware split verkleint leakage maar sluit het niet uit. Er is risico dat het model achtergrondkenmerken leert; de kleur- en helderheidsanalyse en GradCAM op misclassificaties geven een eerste indicatie. Een systematische achtergrond-segmentatie valt buiten de scope.


## 18. Conclusie

Dit project onderzocht multi-label classificatie van koraalcondities met de Koh Tao dataset [5, 16]. De verbeterde CNN from scratch met vijf convolutieblokken en de drie transfer learning modellen zijn vergeleken via een iteratief proces met drie ablations.

Asymmetric Loss is gebruikt als verliesfunctie omdat het de positief-negatief onbalans in multi-label classificatie direct aanpakt [25]. Full fine-tuning is toegepast, omdat de domeinafstand tussen ImageNet en onderwaterfotografie groot is en een frozen backbone aantoonbaar slechter presteert op de Koh Tao dataset [16]. De ablations testen het effect van learning rate, resolutie en augmentatie. De robuustheidstest met drie seeds geeft een betrouwbaarder beeld van de prestaties. De GradCAM analyse op misclassificaties laat zien welke visuele patronen het model misleiden [19].

Per-label drempels verbeteren het resultaat verder zonder opnieuw te hoeven trainen [18]. De testresultaten en per-label scores staan in §15.

Vervolgwerk: testen op andere riffen, stressorlabels toevoegen, ASL hyperparameters tunen voor dit specifieke domein.

## Referenties

[1] T. P. Hughes et al. 2010. Rising to the challenge of sustaining coral reef resilience. *Trends in Ecology and Evolution* 25(11), 633-642.

[2] D. Souter et al. 2021. Status of Coral Reefs of the World: 2020. *GCRMN*.

[3] NOAA Coral Reef Watch. 2024. Fourth global coral bleaching event. https://coralreefwatch.noaa.gov

[4] M. Moniruzzaman et al. 2017. Deep Learning on Underwater Marine Object Detection: A Survey. In *ACIVS 2017, LNCS* 10617, Springer, 150-160.

[5] X. Shao et al. 2024. Deep Learning for Multilabel Classification of Coral Reef Conditions in the Indo-Pacific. *Aquatic Conservation: Marine and Freshwater Ecosystems* 34(9), e4241.

[6] O. Beijbom et al. 2012. Automated annotation of coral reef survey images. In *IEEE CVPR*.

[7] A. Shihavuddin et al. 2013. Image-Based Coral Reef Classification and Thematic Mapping. *Remote Sensing* 5(4), 1809-1841.

[8] A. Gomez-Rios et al. 2019. Towards highly accurate coral texture images classification using deep CNNs and data augmentation. *Expert Systems With Applications* 118, 315-328.

[9] J. Borbon et al. 2021. Coral Health Identification using Image Classification and CNNs. In *IEEE HNICEM*.

[10] M. Tan and Q. Le. 2019. EfficientNet: Rethinking Model Scaling for CNNs. In *ICML*.

[11] K. He et al. 2016. Deep Residual Learning for Image Recognition. In *IEEE CVPR*.

[12] G. Huang et al. 2017. Densely Connected Convolutional Networks. In *IEEE CVPR*.

[13] M. L. Zhang and Z. H. Zhou. 2014. A review on multi-label learning algorithms. *IEEE TKDE* 26(8), 1819-1837.

[14] A. Tharwat. 2021. Classification assessment methods. *Applied Computing and Informatics* 17(1), 168-192.

[15] K. Chen et al. 2021. A new deep learning engine for CoralNet. In *IEEE/CVF ICCV Workshops*.

[16] X. Shao et al. 2025. Multi-label classification for multi-temporal, multi-spatial coral reef condition monitoring. *Marine Pollution Bulletin* 215, 117878.

[17] K. Sechidis et al. 2011. On the Stratification of Multi-Label Data. In *ECML PKDD*, 145-158.

[18] Z. C. Lipton et al. 2014. Optimal Thresholding of Classifiers to Maximize F1 Measure. In *ECML PKDD*, 225-239.

[19] R. R. Selvaraju et al. 2017. Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization. In *IEEE ICCV*.

[20] PyTorch. Automatic Mixed Precision — torch.cuda.is_bf16_supported. https://pytorch.org/docs/stable/generated/torch.cuda.is_bf16_supported.html

[21] I. Loshchilov and F. Hutter. 2019. Decoupled Weight Decay Regularization. In *ICLR*.

[22] PyTorch. torch.optim.AdamW. https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html

[23] PyTorch. Transfer Learning for Computer Vision Tutorial. https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

[24] T. Lin et al. 2017. Focal Loss for Dense Object Detection. In *IEEE ICCV*.

[25] T. Ridnik et al. 2021. Asymmetric Loss For Multi-Label Classification. In *IEEE/CVF ICCV*, 82-91.

[26] C. Zauner. 2010. Implementation and Benchmarking of Perceptual Image Hash Functions. Master's thesis, *Upper Austria University of Applied Sciences, Hagenberg Campus*.

[27] N. Srivastava et al. 2014. Dropout: A Simple Way to Prevent Neural Networks from Overfitting. *Journal of Machine Learning Research* 15, 1929-1958.

[28] R. Pascanu et al. 2013. On the Difficulty of Training Recurrent Neural Networks. In *ICML*, 1310-1318.

[29] I. Loshchilov and F. Hutter. 2017. SGDR: Stochastic Gradient Descent with Warm Restarts. In *ICLR*.

[30] PyTorch. Models and pre-trained weights — Torchvision documentation. https://pytorch.org/vision/stable/models.html
